# GRC Security Intelligence - Phase 1: Synthetic source data

Writes raw files for 10 simulated sources to `/Volumes/<catalog>/grc_bronze/landing/` and generator ground truth to `<catalog>.grc_synth` (`catalog` job parameter / widget, default `workspace`) (demo only, not for the pipeline). Window **2026-07-29 .. 2026-10-27**, ACME Corp, 5 BUs.

| Folder | Source system | Format |
| --- | --- | --- |
| `hr/` | Workday workers | CSV |
| `okta/users`, `okta/groups`, `okta/group_memberships`, `okta/system_log` | Okta IdP | JSON / Parquet / JSONL (partitioned by day) |
| `servicenow/cmdb_ci` | ServiceNow CMDB (Lakeflow Connect sim) | CSV |
| `servicenow/change_request`, `sn_si_incident`, `sn_vul_vulnerable_item` | ServiceNow ITSM / SecOps | CSV |
| `aws/config_snapshot`, `aws/cloudtrail` | AWS Config + CloudTrail (`{"Records":[...]}`) | JSON |
| `tenable/vuln_export` | Tenable weekly scans (partitioned by scan_date) | CSV |
| `crowdstrike/devices`, `crowdstrike/alerts` | CrowdStrike EDR | JSON |
| `lms/enrollments`, `lms/phishing_results` | KnowBe4-style LMS | CSV |
| `tprm/vendor_assessments` | Third-party risk | CSV |
| `sharepoint/lists/*`, `sharepoint/documents/Policies/*.pdf`, `sharepoint/documents_metadata` | SharePoint (Lakeflow Connect sim) | JSON + PDF |

**Injected story**
* Sep-15 restructuring (Sales/Ops): 48 IdP accounts not deprovisioned, 11 keep signing in (some from unusual countries).
* Oct-05 Finance legacy ERP MFA exemption group: 65 Finance users lose MFA (incl. ERP admins); covering exception EXC-003 expired 2026-09-30.
* Sep-30 KEV CVE-2026-31337 (OpenGateway RCE) on 297 Linux servers; 141 still open on the Oct-26 scan.
* Oct-08 CloudTrail StopLogging in sandbox-ops; Oct-12 S3 bucket acme-eng-ml-training-exports made public with no change ticket.
* Oct-14..18 ransomware precursor on Ops Windows hosts (620 alerts, triage far outside SLA) -> SEV1 INC0012045 contained in 30h (SLA 4h).
* Oct-20 root console login without MFA from Tor exit node in prod-payments.
* Engineering: only ~80% of prod IAM/infra changes ticketed; 45% of Eng privileged memberships missed the Q3 access review.
* Sales training completion 80% (target 95%); 3 critical vendors with assessments > 12 months.
* SharePoint: POL-002 register says reviewed 2026-06-15 but PDF approval date is 2025-08-01; POL-004 PDF states 30-day Critical SLA (standard is 15).

Runs end-to-end in ~6 minutes on serverless. Helpers live in `gen_helpers.py` in this folder.

In [0]:
%pip install -q fpdf2

In [0]:
import os, math, json, hashlib, datetime as dt
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("as_of_date", "2026-10-27")
CAT = dbutils.widgets.get("catalog")
# The synthetic story is fixed to the window 2026-07-29 .. 2026-10-27 (as_of_date default)
# gen_helpers.py sits next to this notebook (bundle: src/setup/)
exec(open(os.path.join(os.getcwd(), "gen_helpers.py")).read())
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SYN} COMMENT 'Synthetic data generator ground truth (demo only)'")

In [0]:
FIRST = ["James","Mary","Robert","Patricia","John","Jennifer","Michael","Linda","David","Elizabeth","William","Barbara","Richard","Susan","Joseph","Jessica","Thomas","Sarah","Carlos","Karen","Daniel","Lisa","Matthew","Nancy","Anthony","Priya","Mark","Sandra","Wei","Ashley","Ahmed","Emily","Kenji","Olivia","Luca","Sofia","Raj","Chloe","Mateo","Hannah","Oliver","Amara","Noah","Fatima","Lukas","Ines","Arjun","Mei","Diego","Zoe"]
LAST = ["Smith","Johnson","Williams","Brown","Jones","Garcia","Miller","Davis","Rodriguez","Martinez","Hernandez","Lopez","Gonzalez","Wilson","Anderson","Thomas","Taylor","Moore","Jackson","Martin","Lee","Perez","Thompson","White","Harris","Clark","Lewis","Robinson","Walker","Young","Patel","Nguyen","Kim","Chen","Muller","Schmidt","Rossi","Silva","Tanaka","Kowalski","Novak","Dubois","Murphy","OBrien","Singh","Khan","Ivanova","Larsen","Costa","Fischer"]
BU_SHORT = {"BU-CORP":"corp","BU-FIN":"fin","BU-ENG":"eng","BU-SAL":"sal","BU-OPS":"ops"}
TITLES = {"BU-CORP":["IT Support Analyst","Systems Administrator","IT Manager","Network Engineer","Service Desk Lead"],
          "BU-FIN":["Accountant","AP Specialist","Financial Analyst","Controller","Treasury Analyst","Payroll Specialist"],
          "BU-ENG":["Software Engineer","Senior Software Engineer","DevOps Engineer","Data Engineer","SRE","Engineering Manager","ML Engineer"],
          "BU-SAL":["Account Executive","Sales Development Rep","Marketing Specialist","Sales Manager","Solutions Consultant"],
          "BU-OPS":["Operations Analyst","Logistics Coordinator","Plant Technician","Operations Manager","Supply Chain Planner"]}
LOC = {"BU-CORP":["London","Dublin","Berlin"],"BU-FIN":["New York","London","Singapore"],"BU-ENG":["San Francisco","Austin","Toronto"],
       "BU-SAL":["Chicago","Austin","New York"],"BU-OPS":["Singapore","Sydney","Bangalore"]}

# ---------------- HR workers (Workday-style) ----------------
N_EMP = 4000
e = spark.range(N_EMP, numPartitions=8).withColumnRenamed("id", "idx")
e = e.withColumn("bu_id", wpick([("BU-CORP",.12),("BU-FIN",.26),("BU-ENG",.56),("BU-SAL",.82),("BU-OPS",1.0)], u("idx","bu")))
e = (e.withColumn("employee_id", F.concat(F.lit("W"), F.lpad(F.col("idx").cast("string"), 6, "0")))
      .withColumn("first_name", pick(FIRST, "idx", "fn")).withColumn("last_name", pick(LAST, "idx", "ln"))
      .withColumn("email", F.lower(F.concat(F.substring("first_name",1,1), F.col("last_name"), F.lpad(F.col("idx").cast("string"),4,"0"), F.lit("@acmecorp.com"))))
      .withColumn("worker_type", F.when(u("idx","wt") < .88, "Employee").otherwise("Contractor")))
e = e.withColumn("job_title", by_key(TITLES, "bu_id", lambda v: pick(v, "idx", "title"))) \
     .withColumn("location", by_key(LOC, "bu_id", lambda v: pick(v, "idx", "loc")))
e = e.withColumn("hire_date", F.when(u("idx","new") < .03, F.date_add(F.lit(START), (u("idx","hd")*80).cast("int")))
                              .otherwise(F.date_sub(F.lit(START), (F.lit(30) + F.pow(u("idx","hd"), 1.6)*3600).cast("int"))))
e = (e.withColumn("is_reorg", F.col("bu_id").isin("BU-SAL","BU-OPS") & (u("idx","reorg") < .10) & (F.col("hire_date") < "2026-09-01"))
      .withColumn("termination_date",
          F.when(F.col("is_reorg"), F.lit("2026-09-15").cast("date"))
           .when(u("idx","term") < .035, F.date_add(F.lit(START), (u("idx","td")*90).cast("int")))
           .when(u("idx","old") < .02, F.date_sub(F.lit(START), (u("idx","od")*200 + 5).cast("int"))))
      .withColumn("termination_date", F.when(F.col("termination_date") > F.col("hire_date"), F.col("termination_date")))
      .withColumn("employment_status", F.when(F.col("termination_date").isNotNull(), "Terminated").otherwise("Active"))
      .withColumn("termination_reason", F.when(F.col("is_reorg"), "Restructuring").when(F.col("termination_date").isNotNull(),
                   pick(["Voluntary","Voluntary","Voluntary","Involuntary","End of Contract"], "idx", "tr"))))
e.write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{SYN}.employees")
emp = spark.table(f"{SYN}.employees")
(emp.select("employee_id","first_name","last_name","email","bu_id","job_title","location","worker_type","hire_date",
            "employment_status","termination_date","termination_reason")
    .withColumn("bu_id", F.when(u("employee_id","hrbu") < .003, F.lit(None)).otherwise(F.col("bu_id")))  # minor DQ noise
    .coalesce(1).write.mode("overwrite").option("header", True).csv(f"{LAND}/hr/workers_snapshot_2026-10-27"))

# ---------------- CMDB assets (ServiceNow cmdb_ci export) ----------------
N_AST = 3000
a = spark.range(N_AST, numPartitions=8).withColumnRenamed("id","idx")
a = a.withColumn("sys_class_name", wpick([("cmdb_ci_linux_server",.33),("cmdb_ci_win_server",.52),("cmdb_ci_computer",.84),("cmdb_ci_db_instance",.94),("cmdb_ci_netgear",1.0)], u("idx","cls")))
a = a.withColumn("bu_id", wpick([("BU-ENG",.35),("BU-OPS",.57),("BU-CORP",.75),("BU-FIN",.87),("BU-SAL",1.0)], u("idx","abu")))
is_srv = F.col("sys_class_name").isin("cmdb_ci_linux_server","cmdb_ci_win_server","cmdb_ci_db_instance")
a = a.withColumn("tier", F.when(F.col("sys_class_name")=="cmdb_ci_computer", "T3")
                        .when(is_srv & (u("idx","tier") < .12), "T0").when(u("idx","tier") < .32, "T1").when(u("idx","tier") < .70, "T2").otherwise("T3"))
a = a.withColumn("environment", F.when(F.col("sys_class_name")=="cmdb_ci_computer","corporate").when(u("idx","env") < .70, "production").otherwise("non-production"))
a = a.withColumn("tier", F.when((F.col("environment")=="non-production") & F.col("tier").isin("T0","T1"), "T2").otherwise(F.col("tier")))
a = a.withColumn("internet_facing", is_srv & (F.col("environment")=="production") & (u("idx","inet") < .14))
cls_short = F.when(F.col("sys_class_name")=="cmdb_ci_linux_server","lnx").when(F.col("sys_class_name")=="cmdb_ci_win_server","win") \
             .when(F.col("sys_class_name")=="cmdb_ci_computer","lt").when(F.col("sys_class_name")=="cmdb_ci_db_instance","db").otherwise("net")
bu_short = F.create_map(*[x for k,v in BU_SHORT.items() for x in (F.lit(k),F.lit(v))])[F.col("bu_id")]
a = a.withColumn("name", F.concat_ws("-", bu_short, cls_short, F.lpad(F.col("idx").cast("string"),4,"0")))
a = a.withColumn("os", F.when(F.col("sys_class_name")=="cmdb_ci_linux_server", pick(["Ubuntu 22.04","Ubuntu 24.04","RHEL 8","RHEL 9","Amazon Linux 2023"],"idx","os"))
                       .when(F.col("sys_class_name")=="cmdb_ci_win_server", pick(["Windows Server 2019","Windows Server 2022","Windows Server 2022"],"idx","os"))
                       .when(F.col("sys_class_name")=="cmdb_ci_computer", F.when(u("idx","os") < .75, "Windows 11").otherwise("macOS 15"))
                       .when(F.col("sys_class_name")=="cmdb_ci_db_instance", pick(["PostgreSQL 15","SQL Server 2022","Oracle 19c","MySQL 8.0"],"idx","os"))
                       .otherwise(pick(["Cisco IOS-XE 17","Palo Alto PAN-OS 11","Juniper Junos 23"],"idx","os")))
ACCTS = {"BU-FIN":["prod-payments","corp-shared"],"BU-ENG":["prod-core","eng-ml","eng-dev"],"BU-OPS":["prod-core","sandbox-ops"],
         "BU-CORP":["corp-shared"],"BU-SAL":["sales-crm-integrations"]}
a = a.withColumn("cloud_account", F.when(is_srv, by_key(ACCTS, "bu_id", lambda v: pick(v, "idx", "acct"))))
a = a.withColumn("ip_address", F.concat_ws(".", F.lit("10"), (F.pmod(F.hash("idx",F.lit("ip1")),F.lit(250))+1).cast("string"),
                                          (F.pmod(F.hash("idx",F.lit("ip2")),F.lit(250))+1).cast("string"), (F.pmod(F.col("idx"),F.lit(250))+2).cast("string")))
a = a.withColumn("data_classification", F.when(F.col("tier")=="T0","Confidential").when(F.col("sys_class_name")=="cmdb_ci_db_instance", F.when(u("idx","dc")<.7,"Confidential").otherwise("Internal"))
                                        .otherwise(F.when(u("idx","dc") < .25, "Confidential").when(u("idx","dc") < .9, "Internal").otherwise("Public")))
a = a.withColumn("install_status", F.when(u("idx","ret") < .03, "Retired").otherwise("Installed"))
a = a.withColumn("has_opengateway", (F.col("sys_class_name")=="cmdb_ci_linux_server") & (u("idx","og") < .30))   # truth only
ew = W.partitionBy("bu_id").orderBy("employee_id")
emp_rn = emp.filter("employment_status='Active'").withColumn("rn", F.row_number().over(ew)) \
            .withColumn("cnt", F.count("*").over(W.partitionBy("bu_id"))).select("bu_id","rn","cnt","email")
cnts = emp_rn.select("bu_id","cnt").distinct()
a = a.join(cnts, "bu_id").withColumn("rn", (F.pmod(F.hash("idx",F.lit("own")), F.col("cnt"))+1).cast("int")) \
     .join(emp_rn.select("bu_id","rn",F.col("email").alias("owned_by")), ["bu_id","rn"]).drop("rn","cnt")
a = a.withColumn("sys_id", F.md5(F.concat(F.lit("ci"), F.col("idx").cast("string"))))
a.write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{SYN}.assets")

ast = spark.table(f"{SYN}.assets")
cmdb = (ast.select("sys_id","name","sys_class_name","os","ip_address",
                   F.when((F.col("bu_id")=="BU-OPS") & (u("idx","nobu") < .07), F.lit(None)).when(u("idx","nobu") < .01, F.lit(None)).otherwise(F.col("bu_id")).alias("u_business_unit"),
                   F.when(u("idx","noown") < .04, F.lit(None)).otherwise(F.col("owned_by")).alias("owned_by"),
                   F.when(u("idx","notier") < .02, F.lit(None)).otherwise(F.col("tier")).alias("u_criticality_tier"),
                   "environment","internet_facing","cloud_account","data_classification","install_status",
                   F.expr("timestamp'2026-10-27 02:00:00' - make_interval(0,0,0, cast(pmod(hash(idx),300) as int),0,0,0)").alias("sys_updated_on")))
dups = (ast.filter(u("idx","dup") < .006)
           .select(F.md5(F.concat(F.lit("dup"), F.col("idx").cast("string"))).alias("sys_id"),"name","sys_class_name","os","ip_address",
                   F.col("bu_id").alias("u_business_unit"),"owned_by",F.col("tier").alias("u_criticality_tier"),"environment","internet_facing",
                   "cloud_account","data_classification","install_status",F.lit("2025-11-02 09:14:00").cast("timestamp").alias("sys_updated_on")))
cmdb.unionByName(dups).coalesce(1).write.mode("overwrite").option("header",True).csv(f"{LAND}/servicenow/cmdb_ci/snapshot_2026-10-27")

In [0]:
emp = spark.table(f"{SYN}.employees")

# ---------------- identities (truth) ----------------
term_ts = F.col("termination_date").cast("timestamp") + F.expr("INTERVAL 17 HOURS")
i = (emp.withColumn("user_id", F.concat(F.lit("00u"), F.substring(F.md5("employee_id"), 1, 17)))
        .withColumn("user_type", F.lit("workforce"))
        .withColumn("created_ts", F.col("hire_date").cast("timestamp") + F.expr("INTERVAL 9 HOURS"))
        .withColumn("is_orphan", F.coalesce(F.col("is_reorg") & (u("idx","orph") < .27), F.lit(False)))
        .withColumn("deprov_delay_h", F.when(F.col("termination_date").isNull() | F.col("is_orphan"), F.lit(None))
                    .when(F.col("termination_date") < F.lit(START), F.lit(2.0))
                    .when(u("idx","late") < .88, 1 + u("idx","dl")*20).otherwise(30 + u("idx","dl")*170))
        .withColumn("deprov_ts", add_hours(term_ts, F.col("deprov_delay_h")))
        .withColumn("deprov_ts", F.when(F.col("deprov_ts") <= F.lit(END_TS).cast("timestamp"), F.col("deprov_ts")))
        .withColumn("status", F.when(F.col("deprov_ts").isNull(), "ACTIVE").otherwise("DEPROVISIONED"))
        .withColumn("is_dormant", (F.col("termination_date").isNull()) & (F.col("hire_date") < F.date_sub(F.lit(START), 30)) & (u("idx","dorm") < .05))
        .withColumn("fin_legacy", (F.col("bu_id")=="BU-FIN") & F.col("termination_date").isNull() & ~F.col("is_dormant") & (u("idx","legacy") < .115))
        .withColumn("mfa_enrolled", F.when(F.col("fin_legacy"), False).otherwise(u("idx","mfa") < .975))
        .withColumn("mfa_factor", F.when(~F.col("mfa_enrolled"), "none")
                    .otherwise(wpick([("fido2",.35),("push_number_match",.80),("sms",.95),("totp",1.0)], u("idx","fac"))))
        .withColumn("orphan_still_active", F.col("is_orphan") & (u("idx","orphlogin") < .30)))
i = i.select("idx","user_id","employee_id","email","first_name","last_name","bu_id","job_title","location","user_type","created_ts",
             "termination_date","is_reorg","is_orphan","orphan_still_active","deprov_delay_h","deprov_ts","status","is_dormant","fin_legacy","mfa_enrolled","mfa_factor")
svc = (spark.range(60).withColumnRenamed("id","sidx")
       .withColumn("app", pick(["etl","backup","cicd","monitoring","crm-sync","erp-batch","scanner","sso-bridge"], "sidx", "app"))
       .select((F.col("sidx")+100000).alias("idx"),
               F.concat(F.lit("00u"), F.substring(F.md5(F.concat(F.lit("svc"),F.col("sidx").cast("string"))),1,17)).alias("user_id"),
               F.lit(None).cast("string").alias("employee_id"),
               F.concat(F.lit("svc-"), F.col("app"), F.lit("-"), F.lpad(F.col("sidx").cast("string"),2,"0"), F.lit("@acmecorp.com")).alias("email"),
               F.lit("Service").alias("first_name"), F.col("app").alias("last_name"),
               pick(["BU-ENG","BU-ENG","BU-CORP","BU-FIN","BU-OPS","BU-SAL"], "sidx", "sbu").alias("bu_id"),
               F.lit("Service Account").alias("job_title"), F.lit(None).cast("string").alias("location"), F.lit("service").alias("user_type"),
               F.lit("2024-03-01 10:00:00").cast("timestamp").alias("created_ts"), F.lit(None).cast("date").alias("termination_date"),
               F.lit(False).alias("is_reorg"), F.lit(False).alias("is_orphan"), F.lit(False).alias("orphan_still_active"),
               F.lit(None).cast("double").alias("deprov_delay_h"), F.lit(None).cast("timestamp").alias("deprov_ts"), F.lit("ACTIVE").alias("status"),
               (u("sidx","sdorm") < .15).alias("is_dormant"), F.lit(False).alias("fin_legacy"), F.lit(False).alias("mfa_enrolled"), F.lit("none").alias("mfa_factor")))
bg = (spark.range(3).select((F.col("id")+200000).alias("idx"), F.concat(F.lit("00ubreakglass0000"), F.col("id").cast("string")).alias("user_id"),
       F.lit(None).cast("string").alias("employee_id"), F.concat(F.lit("breakglass-0"), (F.col("id")+1).cast("string"), F.lit("@acmecorp.com")).alias("email"),
       F.lit("Break").alias("first_name"), F.lit("Glass").alias("last_name"), F.lit("BU-CORP").alias("bu_id"), F.lit("Emergency Admin").alias("job_title"),
       F.lit(None).cast("string").alias("location"), F.lit("admin").alias("user_type"), F.lit("2023-01-10 10:00:00").cast("timestamp").alias("created_ts"),
       F.lit(None).cast("date").alias("termination_date"), F.lit(False).alias("is_reorg"), F.lit(False).alias("is_orphan"), F.lit(False).alias("orphan_still_active"),
       F.lit(None).cast("double").alias("deprov_delay_h"), F.lit(None).cast("timestamp").alias("deprov_ts"), F.lit("ACTIVE").alias("status"),
       F.lit(True).alias("is_dormant"), F.lit(False).alias("fin_legacy"), F.lit(True).alias("mfa_enrolled"), F.lit("fido2").alias("mfa_factor")))
i.unionByName(svc).unionByName(bg).write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{SYN}.identities")
ids = spark.table(f"{SYN}.identities")

# ---------------- groups & memberships ----------------
GROUPS = [  # name, privileged
 ("All-Staff",False),("AWS-ReadOnly",False),("Salesforce-Users",False),("AWS-Prod-Admin",True),("Okta-SuperAdmin",True),("Prod-DB-Admin",True),
 ("GitHub-Org-Admin",True),("ERP-Admin",True),("ERP-AP-Create-Vendor",False),("ERP-AP-Approve-Payment",False),("ERP-GL-Post",False),
 ("ERP-GL-Approve",False),("SN-Change-Implementer",False),("SN-Change-Approver",False),("FIN-Legacy-ERP-MFA-Exempt",False)]
grp = spark.createDataFrame([("00g" + hashlib.md5(g.encode()).hexdigest()[:14], g, p) for g,p in GROUPS], "group_id string, group_name string, is_privileged boolean")
grp.write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{SYN}.groups")
grp = spark.table(f"{SYN}.groups")
grp.select("group_id", F.struct(F.col("group_name").alias("name"), F.concat(F.lit("Okta group "), F.col("group_name")).alias("description")).alias("profile"),
           F.lit("OKTA_GROUP").alias("type"), F.lit("2024-01-15T10:00:00.000Z").alias("created")) \
   .coalesce(1).write.mode("overwrite").json(f"{LAND}/okta/groups")

live = ids.filter("status='ACTIVE'")
wf = live.filter("user_type='workforce'")
bu, r = F.col("bu_id"), lambda s: u("idx", s)
rules = {
 "All-Staff": wf,
 "AWS-ReadOnly": wf.filter(bu.isin("BU-ENG","BU-CORP") & (r("ro") < .45)),
 "Salesforce-Users": wf.filter(bu=="BU-SAL"),
 "AWS-Prod-Admin": wf.filter(((bu=="BU-ENG") & (r("awsadm") < .06)) | ((bu=="BU-CORP") & (r("awsadm") < .10))),
 "Okta-SuperAdmin": live.filter(((bu=="BU-CORP") & (F.col("user_type")=="workforce") & (r("oktaadm") < .02)) | (F.col("user_type")=="admin")),
 "Prod-DB-Admin": wf.filter(((bu=="BU-ENG") & (r("dbadm") < .03)) | ((bu=="BU-FIN") & (r("dbadm") < .01))),
 "GitHub-Org-Admin": wf.filter((bu=="BU-ENG") & (r("ghadm") < .02)),
 "ERP-Admin": wf.filter((bu=="BU-FIN") & ((r("erpadm") < .03) | (F.col("fin_legacy") & (r("legadm") < .22)))),
 "ERP-AP-Create-Vendor": wf.filter((bu=="BU-FIN") & ((r("apc") < .20) | (r("sod1") < .025))),
 "ERP-AP-Approve-Payment": wf.filter((bu=="BU-FIN") & ((r("apa") < .12) | (r("sod1") < .025))),
 "ERP-GL-Post": wf.filter((bu=="BU-FIN") & (r("glp") < .25)),
 "ERP-GL-Approve": wf.filter((bu=="BU-FIN") & (r("gla") < .10)),
 "SN-Change-Implementer": wf.filter(bu.isin("BU-ENG","BU-CORP") & (r("sni") < .25)),
 "SN-Change-Approver": wf.filter(bu.isin("BU-ENG","BU-CORP") & (r("sna") < .08)),
 "FIN-Legacy-ERP-MFA-Exempt": wf.filter(F.col("fin_legacy")),
}
mem = None
for g, df in rules.items():
    d = df.select("idx","user_id","email","bu_id","created_ts").withColumn("group_name", F.lit(g))
    mem = d if mem is None else mem.unionByName(d)
mem = mem.join(grp, "group_name")
mk = F.concat("user_id","group_name")
prev_q = (F.col("is_privileged")) & (((F.col("bu_id")=="BU-ENG") & (u(mk,"rev") < .45)) | (u(mk,"rev") < .12))
mem = (mem.withColumn("added_ts", F.when(F.col("group_name")=="FIN-Legacy-ERP-MFA-Exempt", F.lit("2026-10-05 18:42:00").cast("timestamp"))
                                   .otherwise(F.greatest(F.col("created_ts"), F.lit("2025-01-01").cast("timestamp") + F.expr("INTERVAL 1 DAY") * (u(mk,"add")*500).cast("int"))))
          .withColumn("last_access_review_date", F.when(~F.col("is_privileged"), F.lit(None).cast("date"))
                      .when(prev_q, F.date_add(F.lit("2026-04-10"), (u(mk,"rd")*12).cast("int")))
                      .otherwise(F.date_add(F.lit("2026-08-14"), (u(mk,"rd")*12).cast("int"))))
          .withColumn("reviewed_by", F.when(F.col("last_access_review_date").isNotNull(), pick(["iam-review-bot","jlee0123@acmecorp.com","ppatel0456@acmecorp.com"], "user_id", "rvw"))))
mem.write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{SYN}.memberships")
spark.table(f"{SYN}.memberships").select("group_id","group_name","user_id",F.col("email").alias("login"),"added_ts","last_access_review_date","reviewed_by") \
     .coalesce(2).write.mode("overwrite").parquet(f"{LAND}/okta/group_memberships")

# SoD ruleset + privileged entitlements reference
spark.createDataFrame([("SOD-01","ERP-AP-Create-Vendor","ERP-AP-Approve-Payment","Create vendor and approve payment (fraudulent vendor risk)","High"),
                       ("SOD-02","ERP-GL-Post","ERP-GL-Approve","Post and approve journal entries","High"),
                       ("SOD-03","SN-Change-Implementer","SN-Change-Approver","Implement and approve own production changes","Medium")],
                      "rule_id string, entitlement_a string, entitlement_b string, description string, severity string") \
     .write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{REF}.sod_ruleset")
spark.sql(f"COMMENT ON TABLE {REF}.sod_ruleset IS 'Toxic entitlement combinations used by control test T-IAM-06'")
spark.createDataFrame([(g,) for g,p in GROUPS if p], "group_name string") \
     .write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{REF}.privileged_entitlements")
spark.sql(f"COMMENT ON TABLE {REF}.privileged_entitlements IS 'IdP groups that confer privileged access'")

# ---------------- sign-ins (Okta System Log) ----------------
CITY = {"London":"GB","Dublin":"IE","Berlin":"DE","New York":"US","Singapore":"SG","San Francisco":"US","Austin":"US","Toronto":"CA","Chicago":"US","Sydney":"AU","Bangalore":"IN"}
APPS = {"BU-CORP":["Microsoft 365","ServiceNow","AWS Console","Okta Admin"],"BU-FIN":["Microsoft 365","Oracle ERP","Workday","Concur"],
        "BU-ENG":["Microsoft 365","GitHub","AWS Console","Datadog","Jira"],"BU-SAL":["Microsoft 365","Salesforce","Gong","Zoom"],
        "BU-OPS":["Microsoft 365","SAP S/4HANA","ServiceNow","Workday"]}
days = spark.sql(f"SELECT explode(sequence(date'{START}', date'{END}')) AS day")
s = ids.filter("user_type IN ('workforce','admin') AND NOT is_dormant OR orphan_still_active").crossJoin(days)
s = s.filter((F.col("day") >= F.to_date("created_ts")) &
             (F.col("termination_date").isNull() | (F.col("day") < F.col("termination_date")) | F.col("orphan_still_active")))
dow = F.dayofweek("day")
p = F.when(F.col("orphan_still_active") & (F.col("day") >= F.col("termination_date")), .30).when(dow.isin(1,7), .08).otherwise(.86)
s = s.filter(u(F.concat("user_id", F.col("day").cast("string")), "on") < p)
s = s.withColumn("n", F.when(u(F.concat("user_id", F.col("day").cast("string")), "n2") < .35, 2).otherwise(1)) \
     .withColumn("k", F.explode(F.sequence(F.lit(1), F.col("n"))))
key = F.concat("user_id", F.col("day").cast("string"), F.col("k").cast("string"))
after_term = F.col("termination_date").isNotNull() & (F.col("day") >= F.col("termination_date"))
s = (s.withColumn("published", ts_in_day("day", key, "t", 6, 14))
      .withColumn("app", by_key(APPS, "bu_id", lambda v: pick(v, key, "app")))
      .withColumn("country", F.when(after_term & (u(key,"geo") < .4), pick(["RO","NG","VN","BR"], key, "cc"))
                              .otherwise(F.coalesce(F.create_map(*[x for k,v in CITY.items() for x in (F.lit(k),F.lit(v))])[F.col("location")], F.lit("US"))))
      .withColumn("result", F.when(u(key,"fail") < .03, "FAILURE").otherwise("SUCCESS"))
      .withColumn("mfa_verified", F.when(F.col("fin_legacy") & (F.col("day") < "2026-10-05"), True).otherwise(F.col("mfa_enrolled")))
      .withColumn("ip", F.concat_ws(".", (F.pmod(F.hash(key,F.lit("a")),F.lit(200))+20).cast("string"), (F.pmod(F.hash("user_id",F.lit("b")),F.lit(250))).cast("string"),
                                    (F.pmod(F.hash("user_id",F.lit("c")),F.lit(250))).cast("string"), (F.pmod(F.hash(key,F.lit("d")),F.lit(250))).cast("string"))))
signin = s.select(
    F.md5(key).alias("uuid"), F.date_format("published", "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'").alias("published"),
    F.lit("user.session.start").alias("eventType"), F.lit("INFO").alias("severity"),
    F.struct(F.col("user_id").alias("id"), F.lit("User").alias("type"), F.col("email").alias("alternateId"),
             F.concat_ws(" ","first_name","last_name").alias("displayName")).alias("actor"),
    F.struct(F.col("ip").alias("ipAddress"), F.struct(F.col("country").alias("country")).alias("geographicalContext"),
             F.struct(pick(["Windows","Mac OS X","iOS","Android"], key, "os").alias("os"), pick(["CHROME","EDGE","SAFARI","FIREFOX"], key, "br").alias("browser")).alias("userAgent")).alias("client"),
    F.struct(F.col("result").alias("result"), F.when(F.col("result")=="FAILURE", F.lit("INVALID_CREDENTIALS")).alias("reason")).alias("outcome"),
    F.struct(F.lit("OKTA_AUTHENTICATION_PROVIDER").alias("authenticationProvider"),
             F.when(F.col("mfa_verified"), F.col("mfa_factor")).otherwise(F.lit("PASSWORD")).alias("credentialType"),
             F.col("mfa_verified").alias("mfaVerified")).alias("authenticationContext"),
    F.array(F.struct(F.col("app").alias("displayName"), F.lit("AppInstance").alias("type"))).alias("target"),
    F.col("day").alias("event_date"))
signin.repartition("event_date").write.mode("overwrite").partitionBy("event_date").json(f"{LAND}/okta/system_log")

# ---------------- users export (Okta /api/v1/users + factors) ----------------
last = spark.read.json(f"{LAND}/okta/system_log").filter("outcome.result='SUCCESS'") \
            .groupBy(F.col("actor.id").alias("user_id")).agg(F.max("published").alias("lastLogin"))
users = ids.join(last, "user_id", "left").withColumn("lastLogin", F.coalesce(F.col("lastLogin"),
            F.when(F.col("is_dormant"), F.date_format(F.timestamp_seconds(F.unix_timestamp(F.lit(START).cast("timestamp")) - ((F.lit(95) + u("idx","ll")*220)*86400).cast("long")), "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"))))
users.select("user_id", F.col("status"),
             F.date_format("created_ts","yyyy-MM-dd'T'HH:mm:ss.SSS'Z'").alias("created"),
             F.date_format(F.coalesce("deprov_ts","created_ts"),"yyyy-MM-dd'T'HH:mm:ss.SSS'Z'").alias("statusChanged"), "lastLogin",
             F.struct(F.col("email").alias("login"), F.col("email").alias("email"), F.col("first_name").alias("firstName"), F.col("last_name").alias("lastName"),
                      F.col("bu_id").alias("department"), F.col("job_title").alias("title"), F.col("employee_id").alias("employeeNumber"), F.col("user_type").alias("userType")).alias("profile"),
             F.when(F.col("mfa_enrolled"), F.array(F.struct(F.col("mfa_factor").alias("factorType"), F.lit("ACTIVE").alias("status")))).otherwise(F.array()).alias("factors")) \
     .withColumnRenamed("user_id","id").coalesce(1).write.mode("overwrite").json(f"{LAND}/okta/users")

In [0]:
acc = spark.createDataFrame(ACCOUNTS, "account_id string, account_name string, acct_bu string, acct_env string")
ISO = lambda c: F.date_format(c, "yyyy-MM-dd'T'HH:mm:ss'Z'")
SNAP = F.lit("2026-10-27 23:00:00").cast("timestamp")

# ---------------- AWS Config: S3 buckets ----------------
PURP = ["logs","backups","data-lake","exports","artifacts","static","reports","raw","archive","tmp"]
b = (spark.range(238).withColumnRenamed("id","idx")
     .withColumn("account_name", pick([a[1] for a in ACCOUNTS] + ["prod-core","prod-data","eng-ml"], "idx", "bacct"))
     .join(acc, "account_name")
     .withColumn("resourceName", F.concat_ws("-", F.lit("acme"), "account_name", pick(PURP,"idx","purp"), F.lpad(F.col("idx").cast("string"),3,"0")))
     .withColumn("region", pick(REGIONS, "idx", "breg"))
     .withColumn("classification", wpick([("Confidential",.35),("Internal",.85),("Public",1.0)], u("idx","bcls")))
     .withColumn("encryption", F.when((F.col("acct_env")=="non-production") & (u("idx","benc") < .14), "NONE")
                               .when(u("idx","benc") < .02, "NONE").when(u("idx","kms") < .6, "aws:kms").otherwise("AES256"))
     .withColumn("pab", ~((F.col("acct_env")=="non-production") & (u("idx","pab") < .10)))
     .withColumn("is_public", F.lit(False)).withColumn("logging", u("idx","blog") < .8))
story_b = spark.createDataFrame([
    (9001, "eng-ml", "acme-eng-ml-training-exports", "us-east-1", "Confidential", "AES256", False, True, False),
    (9002, "sales-crm-integrations", "acme-sal-marketing-public-assets", "us-east-1", "Public", "AES256", False, True, True)],
    "idx long, account_name string, resourceName string, region string, classification string, encryption string, pab boolean, is_public boolean, logging boolean").join(acc, "account_name")
b.unionByName(story_b).write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{SYN}.s3_buckets")
buckets = spark.table(f"{SYN}.s3_buckets")
cfg_s3 = buckets.select(F.lit("AWS::S3::Bucket").alias("resourceType"), F.concat(F.lit("arn:aws:s3:::"), "resourceName").alias("ARN"),
    F.col("resourceName").alias("resourceId"), "resourceName", F.col("account_id").alias("awsAccountId"), F.col("region").alias("awsRegion"),
    ISO(F.when(F.col("resourceName")=="acme-eng-ml-training-exports", F.lit("2026-10-12 16:49:00").cast("timestamp")).otherwise(SNAP - F.expr("INTERVAL 2 DAYS"))).alias("configurationItemCaptureTime"),
    F.create_map(F.lit("bu"), F.col("acct_bu"), F.lit("data_classification"), F.col("classification")).alias("tags"),
    F.struct(F.when(F.col("encryption")!="NONE", F.col("encryption")).alias("sseAlgorithm"), F.col("pab").alias("publicAccessBlockEnabled"),
             F.col("is_public").alias("aclGrantsAllUsers"), F.col("logging").alias("serverAccessLoggingEnabled")).alias("configuration"))

# ---------------- AWS Config: RDS from CMDB db instances ----------------
dbs = spark.table(f"{SYN}.assets").filter("sys_class_name='cmdb_ci_db_instance' AND install_status='Installed'") \
           .join(acc, F.col("cloud_account")==F.col("account_name"))
dbs = (dbs.withColumn("encrypted", ~(u("idx","renc") < .03))
          .withColumn("backup_fail", F.col("tier").isin("T0","T1") & (u("idx","bfail") < .09))
          .withColumn("last_backup", F.when(F.col("backup_fail"), SNAP - F.expr("INTERVAL 1 DAY") * (3 + (u("idx","bd")*6).cast("int")))
                                     .otherwise(SNAP - F.expr("INTERVAL 1 HOUR") * (1 + (u("idx","bd")*20).cast("int"))))
          .withColumn("restore_test", F.when(F.col("tier").isin("T0","T1"), F.date_sub(F.lit(END), (u("idx","rt")*150).cast("int")))))
dbs.select("sys_id","name","tier","bu_id","account_id","encrypted","backup_fail","last_backup","restore_test").write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{SYN}.rds")
cfg_rds = dbs.select(F.lit("AWS::RDS::DBInstance").alias("resourceType"),
    F.concat(F.lit("arn:aws:rds:us-east-1:"), "account_id", F.lit(":db:"), "name").alias("ARN"), F.col("name").alias("resourceId"), F.col("name").alias("resourceName"),
    F.col("account_id").alias("awsAccountId"), F.lit("us-east-1").alias("awsRegion"), ISO(SNAP - F.expr("INTERVAL 1 DAY")).alias("configurationItemCaptureTime"),
    F.create_map(F.lit("bu"), F.col("bu_id"), F.lit("data_classification"), F.col("data_classification"), F.lit("cmdb_sys_id"), F.col("sys_id"), F.lit("tier"), F.col("tier")).alias("tags"),
    F.struct(F.col("encrypted").alias("storageEncrypted"), F.lit(7).alias("backupRetentionPeriod"), ISO("last_backup").alias("latestRestorableTime"),
             F.col("restore_test").cast("string").alias("lastRestoreTestDate"), F.col("os").alias("engine")).alias("configuration"))

# ---------------- AWS Config: CloudTrail trails ----------------
trails = spark.createDataFrame([(a[0], a[1], r) for a in ACCOUNTS for r in REGIONS], "account_id string, account_name string, region string") \
    .withColumn("isLogging", ~((F.col("account_name")=="sandbox-ops") & (F.col("region")=="us-east-1"))) \
    .withColumn("latestDelivery", F.when(F.col("isLogging"), SNAP - F.expr("INTERVAL 7 MINUTES")).otherwise(F.lit("2026-10-08 14:22:31").cast("timestamp")))
cfg_trail = trails.select(F.lit("AWS::CloudTrail::Trail").alias("resourceType"),
    F.concat(F.lit("arn:aws:cloudtrail:"), "region", F.lit(":"), "account_id", F.lit(":trail/org-trail")).alias("ARN"), F.lit("org-trail").alias("resourceId"),
    F.concat(F.lit("org-trail-"), "region").alias("resourceName"), F.col("account_id").alias("awsAccountId"), F.col("region").alias("awsRegion"),
    ISO(F.when(F.col("isLogging"), SNAP).otherwise(F.lit("2026-10-08 14:22:31").cast("timestamp"))).alias("configurationItemCaptureTime"),
    F.create_map(F.lit("account_name"), F.col("account_name")).alias("tags"),
    F.struct(F.col("isLogging").alias("isLogging"), ISO("latestDelivery").alias("latestDeliveryTime"), F.lit(True).alias("isMultiRegionTrail")).alias("configuration"))
cfg_s3.unionByName(cfg_rds, allowMissingColumns=True).unionByName(cfg_trail, allowMissingColumns=True) \
      .coalesce(1).write.mode("overwrite").json(f"{LAND}/aws/config_snapshot/2026-10-27")

# ---------------- CloudTrail ----------------
admins = spark.table(f"{SYN}.memberships").filter("group_name='AWS-Prod-Admin'").select("email","bu_id") \
              .withColumn("rn", F.row_number().over(W.orderBy("email"))-1)
N_ADM = admins.count()
SVC_ROLES = ["svc-cicd-deployer","svc-backup","svc-monitoring","svc-etl","LambdaExecRole","eks-nodegroup-role","datadog-integration"]
READ = [("ec2.amazonaws.com","DescribeInstances"),("ec2.amazonaws.com","DescribeSecurityGroups"),("s3.amazonaws.com","ListBuckets"),("s3.amazonaws.com","GetBucketAcl"),
        ("iam.amazonaws.com","ListRoles"),("iam.amazonaws.com","GetRole"),("rds.amazonaws.com","DescribeDBInstances"),("sts.amazonaws.com","AssumeRole"),
        ("cloudwatch.amazonaws.com","GetMetricData"),("lambda.amazonaws.com","ListFunctions"),("kms.amazonaws.com","Decrypt"),("logs.amazonaws.com","DescribeLogGroups")]
WRITE = [("ec2.amazonaws.com","RunInstances"),("ec2.amazonaws.com","CreateTags"),("lambda.amazonaws.com","UpdateFunctionCode"),("ecs.amazonaws.com","UpdateService"),
         ("autoscaling.amazonaws.com","UpdateAutoScalingGroup"),("ec2.amazonaws.com","TerminateInstances")]
SEC = [("iam.amazonaws.com","AttachRolePolicy"),("iam.amazonaws.com","PutRolePolicy"),("iam.amazonaws.com","CreateUser"),("iam.amazonaws.com","CreateAccessKey"),
       ("ec2.amazonaws.com","AuthorizeSecurityGroupIngress"),("s3.amazonaws.com","PutBucketPolicy"),("rds.amazonaws.com","ModifyDBInstance"),("kms.amazonaws.com","DisableKey")]
def pick_pair(lst, col, salt):
    arr = F.array(*[F.struct(F.lit(s).alias("src"), F.lit(n).alias("name")) for s, n in lst])
    return F.element_at(arr, (F.pmod(F.hash(col, F.lit(salt)), F.lit(len(lst))) + 1).cast("int"))

ct = (spark.range(150000, numPartitions=16).withColumnRenamed("id","eid")
      .withColumn("day", F.date_add(F.lit(START), (u("eid","day")*91).cast("int")))
      .withColumn("eventTime", ts_in_day("day", "eid", "t", 0, 24))
      .withColumn("account_name", wpick([("prod-core",.30),("prod-data",.45),("prod-payments",.58),("eng-ml",.70),("eng-dev",.82),("sales-crm-integrations",.88),("corp-shared",.96),("sandbox-ops",1.0)], u("eid","acct")))
      .join(F.broadcast(acc), "account_name")
      .withColumn("region", F.when(u("eid","reg") < .8, "us-east-1").otherwise("eu-west-1"))
      .withColumn("cat", F.when(u("eid","cat") < .885, "read").when(u("eid","cat") < .985, "write").otherwise("sec"))
      .withColumn("ev", F.when(F.col("cat")=="read", pick_pair(READ,"eid","r")).when(F.col("cat")=="write", pick_pair(WRITE,"eid","w")).otherwise(pick_pair(SEC,"eid","s")))
      .withColumn("is_human", u("eid","hum") < F.when(F.col("cat")=="sec", .85).otherwise(.25))
      .withColumn("rn", F.pmod(F.hash("eid",F.lit("adm")), F.lit(N_ADM)))
      .join(F.broadcast(admins), "rn", "left")
      .withColumn("svc_role", pick(SVC_ROLES, "eid", "svc")))
ct = (ct.withColumn("principal_type", F.when(F.col("is_human"), "AssumedRole-SSO").otherwise("AssumedRole-Service"))
        .withColumn("arn", F.when(F.col("is_human"), F.concat(F.lit("arn:aws:sts::"), "account_id", F.lit(":assumed-role/AWSReservedSSO_AdministratorAccess_7d1c/"), "email"))
                    .otherwise(F.concat(F.lit("arn:aws:sts::"), "account_id", F.lit(":assumed-role/"), "svc_role", F.lit("/session-"), (F.pmod("eid",F.lit(997))).cast("string"))))
        .withColumn("actor", F.when(F.col("is_human"), F.col("email")).otherwise(F.col("svc_role")))
        .withColumn("mfa", F.when(F.col("is_human"), "true").otherwise("false"))
        .withColumn("src_ip", F.when(F.col("is_human"), F.concat(F.lit("203.0.113."), F.pmod(F.hash("email"),F.lit(250)).cast("string"))).otherwise(F.lit("AWS Internal")))
        .withColumn("errorCode", F.when(u("eid","err") < .02, F.lit("AccessDenied")))
        .withColumn("target", F.concat(F.lit("res-"), F.pmod(F.hash("eid",F.lit("tgt")),F.lit(400)).cast("string")))
        .withColumn("eventID", F.expr("uuid()")))
cl = (admins.crossJoin(spark.sql(f"SELECT explode(sequence(date'{START}', date'{END}')) day"))
      .filter((~F.dayofweek("day").isin(1,7)) & (u(F.concat("email", F.col("day").cast("string")),"cl") < .5))
      .withColumn("eid", F.abs(F.hash("email","day")).cast("long"))
      .withColumn("account_name", F.when(F.col("bu_id")=="BU-CORP", "corp-shared").otherwise(pick(["prod-core","prod-data","eng-ml","eng-dev"], F.concat("email",F.col("day").cast("string")), "cla")))
      .join(F.broadcast(acc), "account_name")
      .select("eid","day", ts_in_day("day", F.concat("email",F.col("day").cast("string")), "clt", 7, 11).alias("eventTime"), "account_name","account_id",
              F.lit("us-east-1").alias("region"), F.lit("login").alias("cat"),
              F.struct(F.lit("signin.amazonaws.com").alias("src"), F.lit("ConsoleLogin").alias("name")).alias("ev"),
              F.lit("AssumedRole-SSO").alias("principal_type"),
              F.concat(F.lit("arn:aws:sts::"), "account_id", F.lit(":assumed-role/AWSReservedSSO_AdministratorAccess_7d1c/"), "email").alias("arn"),
              F.col("email").alias("actor"), F.lit("true").alias("mfa"), F.concat(F.lit("203.0.113."), F.pmod(F.hash("email"),F.lit(250)).cast("string")).alias("src_ip"),
              F.lit(None).cast("string").alias("errorCode"), F.lit(None).cast("string").alias("target"), F.expr("uuid()").alias("eventID")))
base_cols = ["eid","day","eventTime","account_name","account_id","region","cat","ev","principal_type","arn","actor","mfa","src_ip","errorCode","target","eventID"]
def story(rows):
    df = spark.createDataFrame(rows, "ts string, account_name string, region string, src string, name string, principal_type string, actor string, mfa string, src_ip string, target string, cat string")
    return (df.join(acc, "account_name")
              .select(F.lit(None).cast("long").alias("eid"), F.to_date("ts").alias("day"), F.col("ts").cast("timestamp").alias("eventTime"), "account_name","account_id","region","cat",
                      F.struct(F.col("src"), F.col("name")).alias("ev"), "principal_type",
                      F.when(F.col("principal_type")=="Root", F.concat(F.lit("arn:aws:iam::"), "account_id", F.lit(":root")))
                       .otherwise(F.concat(F.lit("arn:aws:sts::"), "account_id", F.lit(":assumed-role/AWSReservedSSO_AdministratorAccess_7d1c/"), "actor")).alias("arn"),
                      "actor","mfa","src_ip", F.lit(None).cast("string").alias("errorCode"), "target", F.expr("uuid()").alias("eventID")))
eng_admin = admins.filter("bu_id='BU-ENG'").orderBy("email").first()["email"]
corp_admin = admins.filter("bu_id='BU-CORP'").orderBy("email").first()["email"]
st = story([
  ("2026-10-08 14:22:31","sandbox-ops","us-east-1","cloudtrail.amazonaws.com","StopLogging","AssumedRole-SSO",corp_admin,"true","203.0.113.77","org-trail","sec"),
  ("2026-10-12 16:44:02","eng-ml","us-east-1","s3.amazonaws.com","DeletePublicAccessBlock","AssumedRole-SSO",eng_admin,"true","198.51.100.23","acme-eng-ml-training-exports","sec"),
  ("2026-10-12 16:47:19","eng-ml","us-east-1","s3.amazonaws.com","PutBucketAcl","AssumedRole-SSO",eng_admin,"true","198.51.100.23","acme-eng-ml-training-exports","sec"),
  ("2026-10-20 03:14:07","prod-payments","us-east-1","signin.amazonaws.com","ConsoleLogin","Root","root","false","185.220.101.34",None,"login"),
  ("2026-10-20 03:16:45","prod-payments","us-east-1","iam.amazonaws.com","ListUsers","Root","root","false","185.220.101.34",None,"read"),
  ("2026-10-20 03:18:12","prod-payments","us-east-1","iam.amazonaws.com","CreateAccessKey","Root","root","false","185.220.101.34","root","sec"),
  ("2026-10-20 03:21:55","prod-payments","us-east-1","s3.amazonaws.com","ListBuckets","Root","root","false","185.220.101.34",None,"read"),
  ("2026-10-20 03:24:30","prod-payments","us-east-1","iam.amazonaws.com","AttachUserPolicy","Root","root","false","185.220.101.34","finance-reporting-svc","sec"),
])
allct = ct.select(*base_cols).unionByName(cl.select(*base_cols)).unionByName(st.select(*base_cols))
allct = allct.filter(~((F.col("account_name")=="sandbox-ops") & (F.col("region")=="us-east-1") & (F.col("eventTime") > F.lit("2026-10-08 14:22:31").cast("timestamp"))))
allct.write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{SYN}.cloudtrail")
allct = spark.table(f"{SYN}.cloudtrail")

ev = F.col("ev.name")
rec = F.struct(
  F.lit("1.09").alias("eventVersion"),
  F.struct(F.when(F.col("principal_type")=="Root","Root").otherwise("AssumedRole").alias("type"),
           F.substring(F.md5("arn"),1,21).alias("principalId"), F.col("arn").alias("arn"), F.col("account_id").alias("accountId"),
           F.when(F.col("principal_type")=="Root", F.lit(None)).otherwise(F.col("actor")).alias("userName"),
           F.struct(F.struct(F.col("mfa").alias("mfaAuthenticated"), ISO(F.col("eventTime") - F.expr("INTERVAL 20 MINUTES")).alias("creationDate")).alias("attributes")).alias("sessionContext")).alias("userIdentity"),
  ISO("eventTime").alias("eventTime"), F.col("ev.src").alias("eventSource"), ev.alias("eventName"), F.col("region").alias("awsRegion"),
  F.col("src_ip").alias("sourceIPAddress"),
  F.when(F.col("principal_type")=="AssumedRole-Service", F.lit("aws-sdk-go-v2/1.30.3")).otherwise(F.lit("Mozilla/5.0 (Windows NT 10.0; Win64; x64)")).alias("userAgent"),
  F.col("errorCode").alias("errorCode"),
  F.struct(F.when(F.col("ev.src")=="s3.amazonaws.com", F.col("target")).alias("bucketName"),
           F.when(ev.isin("AttachRolePolicy","PutRolePolicy"), F.col("target")).alias("roleName"),
           F.when(ev.isin("CreateUser","CreateAccessKey","AttachUserPolicy"), F.col("target")).alias("userName"),
           F.when(ev.isin("AttachRolePolicy","AttachUserPolicy"), F.lit("arn:aws:iam::aws:policy/AdministratorAccess")).alias("policyArn"),
           F.when(ev=="AuthorizeSecurityGroupIngress", F.concat(F.lit("sg-"), F.substring(F.md5("eventID"),1,12))).alias("groupId"),
           F.when(ev=="AuthorizeSecurityGroupIngress", F.when(u("eventID","cidr") < .2, "0.0.0.0/0").otherwise("10.0.0.0/8")).alias("cidrIp"),
           F.when(ev=="ModifyDBInstance", F.col("target")).alias("dBInstanceIdentifier"),
           F.when(ev=="StopLogging", F.col("target")).alias("name"),
           F.when(ev=="PutBucketAcl", F.lit("AllUsers:READ")).alias("accessControlList")).alias("requestParameters"),
  F.when(ev=="ConsoleLogin", F.struct(F.lit("Success").alias("ConsoleLogin"))).alias("responseElements"),
  F.when(ev=="ConsoleLogin", F.struct(F.when(F.col("mfa")=="true","Yes").otherwise("No").alias("MFAUsed"), F.lit("https://console.aws.amazon.com/").alias("LoginTo"))).alias("additionalEventData"),
  F.col("eventID").alias("eventID"), (F.col("cat")=="read").alias("readOnly"),
  F.when(ev=="ConsoleLogin","AwsConsoleSignIn").otherwise("AwsApiCall").alias("eventType"), F.lit(True).alias("managementEvent"),
  F.col("account_id").alias("recipientAccountId"))
files = (allct.withColumn("hour_bucket", (F.hour("eventTime")/6).cast("int"))
              .groupBy("account_id","region","day","hour_bucket").agg(F.collect_list(rec).alias("Records")))
(files.withColumn("dt", F.col("day").cast("string")).select("Records","dt")
      .repartition("dt").write.mode("overwrite").partitionBy("dt").json(f"{LAND}/aws/cloudtrail"))

In [0]:
assets = spark.table(f"{SYN}.assets").filter("install_status='Installed'")
fam = (F.when(F.col("sys_class_name")=="cmdb_ci_linux_server","linux").when(F.col("sys_class_name")=="cmdb_ci_win_server","windows")
        .when(F.col("sys_class_name")=="cmdb_ci_db_instance","database").when(F.col("sys_class_name")=="cmdb_ci_netgear","network")
        .when(F.col("os").startswith("macOS"),"macos").otherwise("windows"))
assets = assets.withColumn("family", fam)

# ---------------- CVE / plugin catalog ----------------
PROD = {"linux":["OpenSSL","glibc","Linux Kernel","sudo","OpenSSH","nginx","Apache httpd","Java OpenJDK","Python","curl"],
        "windows":["Windows SMB","Windows Kernel","Microsoft Office","Chrome","Microsoft Edge",".NET Framework","Windows Print Spooler","Remote Desktop Services"],
        "macos":["macOS WebKit","macOS Kernel","Chrome","Zoom Client"],
        "database":["PostgreSQL","Oracle Database","SQL Server","MySQL"],
        "network":["Cisco IOS XE","PAN-OS GlobalProtect","Junos J-Web"]}
cat_rows, k = [], 0
for f, prods in PROD.items():
    for j in range({"linux":18,"windows":16,"macos":6,"database":8,"network":5}[f]):
        k += 1
        cat_rows.append((f, j, f"CVE-{2025 + (k % 2)}-{10000 + (k*7919) % 89999}", prods[j % len(prods)], 150000 + k*37))
cves = spark.createDataFrame(cat_rows, "family string, fidx int, cve string, product string, plugin_id int")
cves = (cves.withColumn("cvss", F.round(F.when(u("cve","sev") < .12, 9 + u("cve","sc")*1.0).when(u("cve","sev") < .45, 7 + u("cve","sc")*1.9)
                                        .when(u("cve","sev") < .85, 4 + u("cve","sc")*2.9).otherwise(2 + u("cve","sc")*1.9), 1))
            .withColumn("kev", (F.col("cvss") >= 7) & (u("cve","kev") < .15))
            .withColumn("published", F.date_sub(F.lit(START), (u("cve","pub")*420).cast("int"))))
story_cve = spark.createDataFrame([("linux", 99, "CVE-2026-31337", "OpenGateway", 199999, 9.8, True)],
                                  "family string, fidx int, cve string, product string, plugin_id int, cvss double, kev boolean") \
                 .withColumn("published", F.lit("2026-09-30").cast("date"))
cves = cves.unionByName(story_cve)
cves = cves.withColumn("severity", F.when(F.col("cvss")>=9,"Critical").when(F.col("cvss")>=7,"High").when(F.col("cvss")>=4,"Medium").otherwise("Low")) \
           .withColumn("plugin_name", F.concat(F.col("product"), F.lit(" "), F.when(F.col("cvss")>=9, "Remote Code Execution").when(F.col("cvss")>=7,"Privilege Escalation")
                                                 .when(F.col("cvss")>=4,"Information Disclosure").otherwise("Denial of Service"), F.lit(" ("), F.col("cve"), F.lit(")")))
cves.write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{SYN}.cve_catalog")
cves = spark.table(f"{SYN}.cve_catalog")
fam_n = cves.filter("fidx < 99").groupBy("family").agg(F.count("*").alias("fam_n"))

# ---------------- findings lifecycle (truth) ----------------
scanned = assets.withColumn("scanned", ~(u("idx","noscan") < F.when(F.col("bu_id")=="BU-OPS", .12).when(F.col("sys_class_name")=="cmdb_ci_computer", .07).otherwise(.03)))
base_n = F.when(F.col("family").isin("linux","windows"), 14).when(F.col("family")=="database", 8).when(F.col("family")=="network", 5).otherwise(6)
fx = (scanned.filter("scanned").withColumn("nf", (F.pow(u("idx","nf"), 2.2) * base_n * 2.5).cast("int") + 1)
      .withColumn("j", F.explode(F.sequence(F.lit(1), F.col("nf")))).join(fam_n, "family")
      .withColumn("fidx", F.pmod(F.hash("idx","j"), F.col("fam_n")).cast("int"))
      .join(cves.filter("fidx < 99"), ["family","fidx"]).dropDuplicates(["idx","cve"]))
story_fx = assets.filter("has_opengateway").join(cves.filter("cve='CVE-2026-31337'"), "family")
FX_COLS = ["idx","sys_id","name","ip_address","bu_id","tier","family","cve","plugin_id","plugin_name","cvss","severity","kev","published"]
fx = fx.select(*FX_COLS).unionByName(story_fx.select(*FX_COLS))
key = F.concat(F.col("idx").cast("string"), F.col("cve"))
med = F.when(F.col("severity")=="Critical", 9).when(F.col("severity")=="High", 22).when(F.col("severity")=="Medium", 55).otherwise(110)
mult = (F.when(F.col("tier")=="T0", .7).when(F.col("tier")=="T3", 1.4).otherwise(1.0) * F.when(F.col("bu_id")=="BU-OPS", 1.7).otherwise(1.0))
z = F.sqrt(-2*F.log(u(key,"z1")+1e-6)) * F.cos(2*math.pi*u(key,"z2"))
fx = (fx.withColumn("first_found", F.when(F.col("cve")=="CVE-2026-31337", F.lit("2026-10-05").cast("date"))
                    .otherwise(F.greatest(F.col("published") + F.expr("INTERVAL 3 DAYS"), F.date_add(F.lit(START), (u(key,"ff")*211 - 120).cast("int")))))
        .withColumn("fix_days", F.when(F.col("cve")=="CVE-2026-31337",
                                       F.when((F.col("bu_id")=="BU-OPS") | (u(key,"ogfix") < .38), F.lit(None)).otherwise((2 + u(key,"ogd")*14).cast("int")))
                    .when(u(key,"never") < .12, F.lit(None))
                    .otherwise(F.greatest(F.lit(1), (F.exp(F.log(med*mult) + .8*z)).cast("int"))))
        .withColumn("remediated_date", F.date_add("first_found", F.col("fix_days")))
        .withColumn("remediated_date", F.when(F.col("remediated_date") <= F.lit(END), F.col("remediated_date")))
        .filter(F.col("first_found") <= F.lit(END))
        .filter(F.col("remediated_date").isNull() | (F.col("remediated_date") >= F.date_sub(F.lit(START), 7))))
fx.write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{SYN}.vuln_findings")
fx = spark.table(f"{SYN}.vuln_findings")

# ---------------- weekly scan exports (Tenable-style) ----------------
scans = spark.sql("SELECT explode(sequence(date'2026-08-03', date'2026-10-26', interval 7 days)) AS scan_date")
snap = (fx.crossJoin(scans).filter((F.col("first_found") <= F.col("scan_date")) &
         (F.col("remediated_date").isNull() | (F.col("remediated_date") > F.date_sub("scan_date", 7))))
        .withColumn("state", F.when(F.col("remediated_date") <= F.col("scan_date"), "Fixed").otherwise("Active")))
port = F.when(F.col("family")=="windows", 445).when(F.col("family")=="database", 5432).when(F.col("family")=="network", 443).otherwise(22)
snap.select(F.col("plugin_id"), "cve", F.col("cvss").alias("cvss3_base_score"), F.col("severity"), F.col("name").alias("hostname"), "ip_address",
            port.alias("port"), F.lit("tcp").alias("protocol"), "plugin_name",
            F.col("first_found").cast("string").alias("first_found"),
            F.when(F.col("state")=="Fixed", F.col("remediated_date")).otherwise(F.col("scan_date")).cast("string").alias("last_found"),
            "state", F.col("kev").alias("cisa_kev"),
            F.concat(F.lit("SCAN-"), F.date_format("scan_date","yyyyMMdd")).alias("scan_id"), F.col("scan_date").cast("string").alias("scan_date")) \
    .repartition("scan_date").write.mode("overwrite").option("header",True).partitionBy("scan_date").csv(f"{LAND}/tenable/vuln_export")

# ---------------- EDR device inventory (CrowdStrike-style) ----------------
ep = assets.filter(F.col("sys_class_name").isin("cmdb_ci_linux_server","cmdb_ci_win_server","cmdb_ci_computer"))
ep = (ep.withColumn("has_agent", ~(u("idx","noedr") < F.when(F.col("bu_id")=="BU-OPS", .18).otherwise(.03)))
        .withColumn("stale", u("idx","stale") < .03)
        .withColumn("contained", (F.col("bu_id")=="BU-OPS") & F.col("family").isin("windows") & (u("idx","cont") < .05)))
ep.write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{SYN}.edr_devices")
ep = spark.table(f"{SYN}.edr_devices").filter("has_agent")
ep.select(F.substring(F.md5(F.concat(F.lit("dev"), F.col("sys_id"))),1,32).alias("device_id"), F.col("name").alias("hostname"),
    F.when(F.col("family")=="linux","Linux").when(F.col("family")=="macos","Mac").otherwise("Windows").alias("platform_name"), F.col("os").alias("os_version"),
    pick(["7.14.18408","7.15.18511","7.16.18605","7.10.17706"], "idx", "av").alias("agent_version"),
    F.date_format(F.lit("2025-02-01 10:00:00").cast("timestamp") + F.expr("INTERVAL 1 DAY") * (u("idx","fs")*400).cast("int"), "yyyy-MM-dd'T'HH:mm:ss'Z'").alias("first_seen"),
    F.date_format(F.when(F.col("stale"), F.lit(END_TS).cast("timestamp") - F.expr("INTERVAL 1 DAY") * (8 + (u("idx","ls")*40).cast("int")))
                   .otherwise(F.lit(END_TS).cast("timestamp") - F.expr("INTERVAL 1 MINUTE") * (u("idx","ls")*600).cast("int")), "yyyy-MM-dd'T'HH:mm:ss'Z'").alias("last_seen"),
    F.when(F.col("contained"), "contained").otherwise("normal").alias("status"),
    F.when(u("idx","rfm") < .01, "yes").otherwise("no").alias("reduced_functionality_mode"),
    F.col("ip_address").alias("local_ip"), F.array(F.concat(F.lit("SensorGroupingTags/"), F.col("bu_id"))).alias("tags")) \
  .coalesce(1).write.mode("overwrite").json(f"{LAND}/crowdstrike/devices/2026-10-27")
dev = spark.read.json(f"{LAND}/crowdstrike/devices/2026-10-27").join(ep.select(F.col("name").alias("hostname"),"bu_id","family","idx"), "hostname") \
           .withColumn("drn", F.row_number().over(W.orderBy("device_id"))-1)
N_DEV = dev.count()

# ---------------- EDR alerts ----------------
TECH = [("Execution","T1059.001","PowerShell"),("Persistence","T1547.001","Registry Run Keys"),("Credential Access","T1003.001","LSASS Memory"),
        ("Discovery","T1087","Account Discovery"),("Lateral Movement","T1021.002","SMB/Windows Admin Shares"),("Defense Evasion","T1562.001","Disable or Modify Tools"),
        ("Command and Control","T1071.001","Web Protocols"),("Impact","T1486","Data Encrypted for Impact"),("Initial Access","T1566.001","Spearphishing Attachment"),
        ("Execution","T1204.002","Malicious File"),("Privilege Escalation","T1068","Exploitation for Privilege Escalation"),("Collection","T1560","Archive Collected Data")]
tech_arr = F.array(*[F.struct(F.lit(a).alias("tactic"), F.lit(b).alias("technique_id"), F.lit(c).alias("technique")) for a,b,c in TECH])
ANALYSTS = ["soc-tier1-amara","soc-tier1-lukas","soc-tier1-mei","soc-tier2-diego","soc-tier2-priya","soc-tier1-noah"]
SLA_H = F.when(F.col("severity")=="Critical",1).when(F.col("severity")=="High",4).when(F.col("severity")=="Medium",24).otherwise(72)
bg_al = (spark.range(9000, numPartitions=8).withColumnRenamed("id","aid")
         .withColumn("drn", F.pmod(F.hash("aid",F.lit("dev")), F.lit(N_DEV))).join(F.broadcast(dev.select("drn","device_id","hostname","bu_id")), "drn")
         .withColumn("created", F.timestamp_seconds(F.unix_timestamp(F.lit(START).cast("timestamp")) + (u("aid","ct")*91*86400).cast("long")))
         .withColumn("severity", wpick([("Critical",.02),("High",.11),("Medium",.45),("Low",1.0)], u("aid","sev")))
         .withColumn("t", F.element_at(tech_arr, (F.pmod(F.hash("aid",F.lit("tech")), F.lit(len(TECH)))+1).cast("int")))
         .withColumn("triage_h", SLA_H * F.exp(F.log(F.lit(.35)) + .9*F.sqrt(-2*F.log(u("aid","z1")+1e-6))*F.cos(2*math.pi*u("aid","z2"))))
         .withColumn("incident_id", F.lit(None).cast("string")))
ops_hosts = dev.filter("bu_id='BU-OPS' AND family='windows'").orderBy("device_id").limit(40).withColumn("orn", F.row_number().over(W.orderBy("device_id"))-1)
STORY_T = [TECH[i] for i in (0,2,4,5,7,3)]
st_arr = F.array(*[F.struct(F.lit(a).alias("tactic"), F.lit(b).alias("technique_id"), F.lit(c).alias("technique")) for a,b,c in STORY_T])
st_al = (spark.range(620).withColumn("aid", F.col("id") + 1000000).withColumn("orn", F.pmod(F.hash("aid",F.lit("oh")), F.lit(40)))
         .join(F.broadcast(ops_hosts.select("orn","device_id","hostname","bu_id")), "orn")
         .withColumn("created", F.timestamp_seconds(F.unix_timestamp(F.lit("2026-10-14 08:00:00").cast("timestamp")) + (F.pow(u("aid","sct"),1.4)*4.5*86400).cast("long")))
         .withColumn("severity", wpick([("Critical",.25),("High",.75),("Medium",1.0)], u("aid","ssev")))
         .withColumn("t", F.element_at(st_arr, (F.pmod(F.hash("aid",F.lit("st")), F.lit(len(STORY_T)))+1).cast("int")))
         .withColumn("triage_h", 2 + u("aid","str")*26)
         .withColumn("incident_id", F.lit("INC0012045")))
al = bg_al.unionByName(st_al.drop("id","orn"), allowMissingColumns=True).drop("drn","orn","id")
al = (al.withColumn("triaged", add_hours(F.col("created"), F.col("triage_h")))
        .withColumn("triaged", F.when(F.col("triaged") <= F.lit(END_TS).cast("timestamp"), F.col("triaged")))
        .withColumn("status", F.when(F.col("triaged").isNull(), "new").when(F.col("incident_id").isNotNull(), "closed")
                    .when(F.col("created") > F.lit("2026-10-25").cast("timestamp"), "in_progress")
                    .when(u("aid","fp") < .55, "closed").otherwise("false_positive")))
al.write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{SYN}.edr_alerts")
al = spark.table(f"{SYN}.edr_alerts")
al.select(F.concat(F.lit("ldt:"), F.substring(F.md5(F.col("aid").cast("string")),1,24)).alias("composite_id"),
          F.date_format("created","yyyy-MM-dd'T'HH:mm:ss'Z'").alias("created_timestamp"),
          F.date_format("triaged","yyyy-MM-dd'T'HH:mm:ss'Z'").alias("first_status_change_timestamp"),
          F.col("severity").alias("severity_name"), F.col("t.tactic").alias("tactic"), F.col("t.technique_id").alias("technique_id"), F.col("t.technique").alias("technique"),
          "status", F.when(F.col("triaged").isNotNull(), pick(ANALYSTS, "aid", "an")).alias("assigned_to_name"),
          F.struct("device_id","hostname").alias("device"), F.col("incident_id").alias("linked_incident"),
          F.to_date("created").cast("string").alias("dt")) \
  .repartition("dt").write.mode("overwrite").partitionBy("dt").json(f"{LAND}/crowdstrike/alerts")

In [0]:
acc = spark.createDataFrame(ACCOUNTS, "account_id string, account_name string, acct_bu string, acct_env string")
TS = lambda c: F.date_format(c, "yyyy-MM-dd HH:mm:ss")
GRP = {"BU-CORP":"IT Infrastructure","BU-FIN":"Finance Applications","BU-ENG":"Platform Engineering","BU-SAL":"Business Systems","BU-OPS":"Operations IT"}
grp_map = F.create_map(*[x for k,v in GRP.items() for x in (F.lit(k),F.lit(v))])

# ---------------- change requests (derived from CloudTrail security changes + generic releases) ----------------
ct = spark.table(f"{SYN}.cloudtrail").filter("cat='sec' AND eid IS NOT NULL").join(acc.select("account_name","acct_bu","acct_env"), "account_name")
p_ticket = F.when(F.col("acct_env")=="non-production", .35).when(F.col("acct_bu")=="BU-ENG", .80).otherwise(.93)
chg = (ct.filter(u("eventID","tkt") < p_ticket)
         .withColumn("start", F.col("eventTime") - F.expr("INTERVAL 1 HOUR") * (1 + (u("eventID","s")*3).cast("int")))
         .withColumn("end", F.col("eventTime") + F.expr("INTERVAL 1 HOUR") * (1 + (u("eventID","e")*3).cast("int")))
         .withColumn("approver", F.when(u("eventID","self") < .05, F.col("actor")).otherwise(pick(["cab-chair@acmecorp.com","jlee0123@acmecorp.com","ppatel0456@acmecorp.com","mkim0789@acmecorp.com"], "eventID", "apr")))
         .select(F.col("eventID").alias("src_event"), F.concat(F.col("ev.name"), F.lit(" on "), F.coalesce("target", F.lit("account")), F.lit(" ("), F.col("account_name"), F.lit(")")).alias("short_description"),
                 F.when(u("eventID","typ") < .7, "Normal").when(u("eventID","typ") < .95, "Standard").otherwise("Emergency").alias("type"),
                 F.col("actor").alias("assigned_to"), "approver", F.col("account_name").alias("u_cloud_account"), F.coalesce("target", F.col("account_name")).alias("cmdb_ci"),
                 "start","end", F.col("acct_bu").alias("bu_id")))
SVCS = ["payments-api","erp-batch","crm-sync","data-platform","vpn","sso"]
generic = (spark.range(900).withColumnRenamed("id","gid")
           .withColumn("day", F.date_add(F.lit(START), (u("gid","d")*91).cast("int"))).withColumn("start", ts_in_day("day","gid","st",18,4))
           .withColumn("end", F.col("start") + F.expr("INTERVAL 2 HOURS"))
           .withColumn("bu_id", wpick([("BU-ENG",.45),("BU-CORP",.65),("BU-FIN",.78),("BU-OPS",.9),("BU-SAL",1.0)], u("gid","bu")))
           .select(F.lit(None).cast("string").alias("src_event"),
                   F.concat(pick(["Release","Patch deployment","Config update","Certificate renewal","DB maintenance"], "gid","what"), F.lit(" - "), pick(SVCS, "gid","svc")).alias("short_description"),
                   F.when(u("gid","typ") < .55, "Standard").otherwise("Normal").alias("type"),
                   pick(["dgarcia0112@acmecorp.com","rnguyen0333@acmecorp.com","asmith0444@acmecorp.com","lchen0555@acmecorp.com"], "gid","as").alias("assigned_to"),
                   pick(["cab-chair@acmecorp.com","jlee0123@acmecorp.com","ppatel0456@acmecorp.com"], "gid","apr").alias("approver"),
                   F.lit(None).cast("string").alias("u_cloud_account"), pick(SVCS, "gid","svc").alias("cmdb_ci"), "start","end","bu_id"))
allchg = chg.unionByName(generic).withColumn("rn", F.row_number().over(W.orderBy("start","short_description")))
allchg.withColumn("number", F.concat(F.lit("CHG"), F.lpad((F.col("rn")+30000).cast("string"), 7, "0"))) \
      .write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{SYN}.changes")
spark.table(f"{SYN}.changes").select("number","short_description","type",
    F.when(F.col("end") < F.lit(END_TS).cast("timestamp"), "Closed").otherwise("Scheduled").alias("state"), F.lit("approved").alias("approval"),
    "assigned_to", F.col("approver").alias("approved_by"), grp_map[F.col("bu_id")].alias("assignment_group"),
    F.col("bu_id").alias("u_business_unit"), "cmdb_ci", "u_cloud_account", TS("start").alias("start_date"), TS("end").alias("end_date"),
    TS(F.col("start") - F.expr("INTERVAL 2 DAYS")).alias("sys_created_on")) \
  .coalesce(1).write.mode("overwrite").option("header",True).csv(f"{LAND}/servicenow/change_request")

# ---------------- security incidents ----------------
story_inc = [
 ("INC0012045","SEV1","Malware","Ransomware precursor activity on Operations Windows hosts (LSASS dumping, SMB lateral movement, encryption attempts)","BU-OPS",
  "2026-10-14 08:05:00","2026-10-16 09:30:00","2026-10-17 15:30:00","2026-10-22 18:00:00",False,"T1486","Containment delayed: 7 hosts without EDR agent required manual isolation"),
 ("INC0012058","SEV2","Data Exposure","S3 bucket acme-eng-ml-training-exports made public (AllUsers READ) without change ticket","BU-ENG",
  "2026-10-12 16:47:19","2026-10-19 10:00:00","2026-10-19 14:00:00",None,False,None,"Detected 7 days after exposure by external researcher report"),
 ("INC0012071","SEV2","Unauthorized Access","Root console login without MFA from Tor exit node in prod-payments; access key created","BU-FIN",
  "2026-10-20 03:14:07","2026-10-20 07:40:00","2026-10-20 11:05:00",None,False,"T1078.004","Root credentials rotated, access key deleted"),
 ("INC0012088","SEV3","Insider / Access","Terminated employee account (restructuring) still signing in from unusual geography","BU-SAL",
  "2026-10-23 09:00:00","2026-10-23 11:20:00","2026-10-23 16:45:00",None,False,"T1078","Account disabled; 47 further orphaned accounts identified"),
]
inc_story = spark.createDataFrame(story_inc, "number string, severity string, category string, short_description string, bu_id string, first_activity string, declared string, contained string, resolved string, pir_completed boolean, mitre string, work_notes string")
CATS = ["Phishing","Malware","Unauthorized Access","Policy Violation","Data Exposure","Lost Device","DDoS"]
inc_bg = (spark.range(26).withColumnRenamed("id","iid")
          .withColumn("declared_ts", F.timestamp_seconds(F.unix_timestamp(F.lit(START).cast("timestamp")) + (u("iid","d")*80*86400).cast("long")))
          .withColumn("severity", wpick([("SEV1",.04),("SEV2",.30),("SEV3",1.0)], u("iid","sev")))
          .withColumn("sla_h", F.when(F.col("severity")=="SEV1",4).when(F.col("severity")=="SEV2",24).otherwise(72))
          .withColumn("contain_h", F.col("sla_h") * F.when(u("iid","br") < .1, 1.5 + u("iid","x")).otherwise(.15 + u("iid","x")*.8))
          .select(F.concat(F.lit("INC00"), (F.col("iid")*37 + 11000).cast("string")).alias("number"), "severity", pick(CATS,"iid","cat").alias("category"),
                  F.concat(pick(CATS,"iid","cat"), F.lit(" - "), pick(["user report","EDR detection","SIEM correlation","email gateway","third-party notification"],"iid","src")).alias("short_description"),
                  pick(["BU-ENG","BU-SAL","BU-FIN","BU-OPS","BU-CORP"],"iid","bu").alias("bu_id"),
                  TS(F.col("declared_ts") - F.expr("INTERVAL 3 HOURS")).alias("first_activity"), TS("declared_ts").alias("declared"),
                  TS(add_hours(F.col("declared_ts"), F.col("contain_h"))).alias("contained"),
                  TS(add_hours(F.col("declared_ts"), F.col("contain_h") + 24 + u("iid","r")*96)).alias("resolved"),
                  (u("iid","pir") < .8).alias("pir_completed"), F.lit(None).cast("string").alias("mitre"), F.lit(None).cast("string").alias("work_notes")))
inc_story.unionByName(inc_bg).write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{SYN}.incidents")
spark.table(f"{SYN}.incidents").select("number","short_description","category", F.col("severity").alias("u_severity"),
    F.when(F.col("resolved").isNotNull(), "Closed").otherwise("Contained").alias("state"),
    F.col("bu_id").alias("u_business_unit"), F.lit("Security Operations").alias("assignment_group"),
    F.col("first_activity").alias("u_first_activity"), F.col("declared").alias("opened_at"), F.col("contained").alias("u_contained_at"),
    F.col("resolved").alias("closed_at"), F.col("pir_completed").alias("u_pir_completed"), F.col("mitre").alias("u_mitre_technique"), "work_notes") \
  .coalesce(1).write.mode("overwrite").option("header",True).option("escape",'"').csv(f"{LAND}/servicenow/sn_si_incident")

# ---------------- vulnerability remediation tasks ----------------
fx = spark.table(f"{SYN}.vuln_findings").filter("severity IN ('Critical','High') AND tier IN ('T0','T1')")
sla_days = F.when(F.col("kev"), 14).when(F.col("severity")=="Critical", 15).otherwise(30) * F.when(F.col("tier")=="T0", .5).otherwise(1.0)
(fx.withColumn("opened", F.date_add("first_found", 1))
   .withColumn("due", F.date_add("opened", sla_days.cast("int")))
   .withColumn("rn", F.row_number().over(W.orderBy("first_found","name","cve")))
   .select(F.concat(F.lit("VIT"), F.lpad((F.col("rn")+50000).cast("string"),7,"0")).alias("number"), "cve", F.col("name").alias("cmdb_ci"),
           F.col("severity").alias("risk_rating"), F.col("tier").alias("u_asset_tier"),
           F.when(F.col("remediated_date").isNotNull(), "Closed").when(u(F.concat("name","cve"),"st") < .6, "In Progress").otherwise("Open").alias("state"),
           grp_map[F.col("bu_id")].alias("assignment_group"),
           F.col("opened").cast("string").alias("opened_at"), F.col("due").cast("string").alias("due_date"),
           F.col("remediated_date").cast("string").alias("closed_at"))
   .coalesce(1).write.mode("overwrite").option("header",True).csv(f"{LAND}/servicenow/sn_vul_vulnerable_item"))

In [0]:
ids = spark.table(f"{SYN}.identities").filter("user_type='workforce' AND status='ACTIVE' AND NOT is_orphan")
w = ids.join(spark.table(f"{SYN}.employees").select("employee_id","hire_date"), "employee_id")

# ---------------- annual security awareness ----------------
p_done = F.when(F.col("bu_id")=="BU-SAL", .81).when(F.col("bu_id")=="BU-OPS", .91).otherwise(.965)
tr = (w.withColumn("assigned", F.when(F.col("hire_date") > "2026-01-08", F.date_add("hire_date", 7)).otherwise(F.lit("2026-01-15").cast("date")))
       .withColumn("due", F.when(F.col("hire_date") > "2026-01-08", F.date_add("assigned", 30)).otherwise(F.lit("2026-03-31").cast("date")))
       .withColumn("done", u("idx","trn") < p_done)
       .withColumn("completed", F.when(F.col("done"), F.least(F.date_add("assigned", (3 + F.pow(u("idx","tc"),1.5)*70).cast("int")), F.lit(END).cast("date"))))
       .withColumn("completed", F.when(F.col("completed") <= F.lit(END), F.col("completed")))
       .withColumn("status", F.when(F.col("completed").isNotNull(), "Passed").when(F.col("due") < F.lit(END), "Past Due")
                   .when(u("idx","ip") < .5, "In Progress").otherwise("Not Started")))
tr.select(F.concat(F.lit("ENR-"), F.substring(F.md5(F.concat("employee_id", F.lit("sa26"))),1,10)).alias("enrollment_id"),
          F.col("email").alias("user_email"), "employee_id", F.lit("Security Awareness Training 2026").alias("course_name"),
          F.col("assigned").cast("string").alias("assigned_date"), F.col("due").cast("string").alias("due_date"),
          F.col("completed").cast("string").alias("completion_date"), "status",
          F.when(F.col("completed").isNotNull(), (78 + u("idx","sc")*22).cast("int")).alias("score")) \
  .coalesce(1).write.mode("overwrite").option("header",True).csv(f"{LAND}/lms/enrollments")

# ---------------- phishing simulations ----------------
camps = spark.createDataFrame([("PH-2026-08","Aug 2026 - Invoice overdue","2026-08-11"),("PH-2026-09","Sep 2026 - Benefits enrollment","2026-09-08"),
                               ("PH-2026-10","Oct 2026 - Shared document (M365)","2026-10-13")], "campaign_id string, campaign_name string, send_date string")
ph = w.crossJoin(camps).withColumn("k", F.concat("employee_id","campaign_id"))
click_p = F.when(F.col("bu_id")=="BU-SAL", .11).when(F.col("bu_id")=="BU-FIN", .08).otherwise(.05) * F.when(F.col("campaign_id")=="PH-2026-10", 1.3).otherwise(1.0)
ph = (ph.withColumn("opened", u("k","op") < .62).withColumn("clicked", F.col("opened") & (u("k","cl") < click_p/.62))
        .withColumn("data_entered", F.col("clicked") & (u("k","de") < .35))
        .withColumn("reported", ~F.col("clicked") & F.col("opened") & (u("k","rp") < F.when(F.col("bu_id")=="BU-ENG", .55).otherwise(.35))))
ph.select("campaign_id","campaign_name","send_date", F.col("email").alias("user_email"), F.lit(True).alias("delivered"),
          "opened","clicked","data_entered","reported").coalesce(1).write.mode("overwrite").option("header",True).csv(f"{LAND}/lms/phishing_results")

# ---------------- third-party vendors ----------------
V = [("PayFlow Payroll Services","Payroll processing","Critical","PII","BU-FIN"),("CloudVault Backup","Backup & DR","Critical","Confidential","BU-CORP"),
     ("DataBridge ETL","Data integration","Critical","Confidential","BU-ENG"),("Okta","Identity provider","Critical","PII","BU-CORP"),
     ("Amazon Web Services","Cloud infrastructure","Critical","Confidential","BU-ENG"),("Salesforce","CRM","Critical","PII","BU-SAL"),
     ("Workday","HRIS","High","PII","BU-CORP"),("ServiceNow","ITSM","High","Confidential","BU-CORP"),("CrowdStrike","EDR","High","Confidential","BU-CORP"),
     ("Tenable","Vulnerability scanning","High","Confidential","BU-CORP"),("GitHub","Source control","High","Confidential","BU-ENG"),
     ("Concur","Expense management","High","PII","BU-FIN"),("Stripe","Payment processing","High","PII","BU-FIN"),("Zoom","Collaboration","High","Internal","BU-CORP"),
     ("Datadog","Observability","High","Confidential","BU-ENG"),("DocuSign","E-signature","High","PII","BU-FIN"),
     ("Gong","Sales intelligence","Medium","PII","BU-SAL"),("HubSpot","Marketing automation","Medium","PII","BU-SAL"),("Slack","Messaging","Medium","Internal","BU-CORP"),
     ("Atlassian","Project tracking","Medium","Internal","BU-ENG"),("Snowplow Analytics","Web analytics","Medium","PII","BU-SAL"),("FreightLink Logistics","Logistics","Medium","Internal","BU-OPS"),
     ("PlantSense IoT","OT monitoring","Medium","Internal","BU-OPS"),("SecurePrint MPS","Managed print","Medium","Internal","BU-CORP"),
     ("TalentHub Recruiting","Recruiting","Medium","PII","BU-CORP"),("Coupa","Procurement","Medium","Confidential","BU-FIN"),
     ("Box","File sharing","Medium","Confidential","BU-CORP"),("Twilio","Messaging API","Medium","PII","BU-ENG"),("Zendesk","Customer support","Medium","PII","BU-SAL"),
     ("Iron Mountain","Records storage","Medium","Confidential","BU-OPS"),("CleanDesk Facilities","Facilities","Low","None","BU-OPS"),
     ("CaterCo","Catering","Low","None","BU-OPS"),("SwagWorks","Promotional goods","Low","None","BU-SAL"),("TravelWise","Travel booking","Low","PII","BU-CORP"),
     ("LegalEase Counsel","Legal services","Low","Confidential","BU-FIN"),("PrintPro","Printing","Low","None","BU-SAL"),("ParkRight","Parking","Low","None","BU-OPS"),
     ("GreenLeaf Plants","Office plants","Low","None","BU-OPS"),("CoffeeCorp","Coffee service","Low","None","BU-OPS"),("SignPost Media","Advertising","Low","Internal","BU-SAL")]
vd = spark.createDataFrame([(f"VND-{i+1:03d}",)+v for i, v in enumerate(V)], "vendor_id string, vendor_name string, service string, criticality string, data_access string, bu_owner string")
overdue = F.col("vendor_name").isin("PayFlow Payroll Services","CloudVault Backup","DataBridge ETL")
spend_mu = F.when(F.col("criticality")=="Critical", 13.5).when(F.col("criticality")=="High", 12.3).when(F.col("criticality")=="Medium", 11.2).otherwise(9.8)
vd = (vd.withColumn("annual_spend_usd", F.round(F.exp(spend_mu + .6*(u("vendor_id","sp")-.5)*2), -2))
        .withColumn("last_assessment_date", F.when(overdue, F.date_sub(F.lit(END), (400 + u("vendor_id","la")*200).cast("int")))
                    .when(F.col("criticality").isin("Critical","High"), F.date_sub(F.lit(END), (20 + u("vendor_id","la")*300).cast("int")))
                    .when(F.col("criticality")=="Medium", F.date_sub(F.lit(END), (60 + u("vendor_id","la")*800).cast("int"))))
        .withColumn("assessment_score", F.when(F.col("last_assessment_date").isNotNull(), F.when(overdue, 55 + u("vendor_id","as")*15).otherwise(68 + u("vendor_id","as")*30)).cast("int"))
        .withColumn("soc2_report_expiry", F.when(F.col("criticality").isin("Critical","High"), F.when(overdue, F.date_sub(F.lit(END), (30 + u("vendor_id","se")*120).cast("int")))
                                                .otherwise(F.date_add(F.lit(END), (20 + u("vendor_id","se")*300).cast("int")))))
        .withColumn("open_findings", F.when(overdue, 3 + (u("vendor_id","of")*5).cast("int")).otherwise((F.pow(u("vendor_id","of"),3)*4).cast("int")))
        .withColumn("contract_security_clause", ~(overdue & (F.col("vendor_name")=="DataBridge ETL")) & (u("vendor_id","csc") < .92) | F.col("criticality").isin("Critical"))
        .withColumn("questionnaire", F.when(F.col("criticality").isin("Critical","High"), "SIG Core").when(F.col("criticality")=="Medium","SIG Lite"))
        .withColumn("relationship_owner", F.concat(F.lower(F.substring("bu_owner",4,4)), F.lit("-vendor-mgr@acmecorp.com"))))
vd.write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{SYN}.vendors")
vt = spark.table(f"{SYN}.vendors")
vt.select(*[F.col(c).cast("string") if c.endswith("date") or c.endswith("expiry") else F.col(c) for c in vt.columns]) \
  .coalesce(1).write.mode("overwrite").option("header",True).csv(f"{LAND}/tprm/vendor_assessments")

In [0]:
from fpdf import FPDF
SP_SITE = "https://acmecorp.sharepoint.com/sites/GRC"
SP = f"{LAND}/sharepoint"

# id, title, owner, version, approved (doc approval), register_last_review (list metadata), mapped controls, key statements
POL = [
 ("POL-001","Information Security Policy","ciso@acmecorp.com","4.2","2026-02-10","2026-02-10",["GRC-POL-01","GRC-RSK-01"],
  ["The CISO owns the information security program and reports quarterly to the Audit Committee.","All policies are reviewed at least every 12 months.","Exceptions require CISO approval and may not exceed 180 days."]),
 ("POL-002","Access Control Policy","iam-lead@acmecorp.com","3.1","2025-08-01","2026-06-15",["GRC-IAM-01","GRC-IAM-02","GRC-IAM-03","GRC-IAM-04","GRC-IAM-05","GRC-IAM-06"],
  ["Multi-factor authentication is mandatory for all workforce identities; privileged accounts must use phishing-resistant MFA.","Accounts of leavers are disabled within 24 hours of HR termination.","Accounts inactive for 90 days are disabled.","Privileged access is recertified every 90 days.","Toxic combinations defined in the SoD ruleset are prohibited."]),
 ("POL-003","Acceptable Use Policy","grc-manager@acmecorp.com","2.4","2026-03-03","2026-03-03",["GRC-TRN-01"],
  ["Users must complete annual security awareness training.","Company data may only be stored on approved services."]),
 ("POL-004","Vulnerability Management Standard","vuln-lead@acmecorp.com","2.0","2025-07-20","2025-07-20",["GRC-VUL-01","GRC-VUL-02","GRC-VUL-03"],
  ["All in-scope assets are scanned at least weekly.","Critical vulnerabilities are remediated within 30 days, High within 60 days, Medium within 90 days.","CISA KEV vulnerabilities are remediated within 14 days."]),
 ("POL-005","Incident Response Plan","soc-manager@acmecorp.com","5.0","2026-05-12","2026-05-12",["GRC-IR-01","GRC-LOG-02"],
  ["SEV1 incidents are contained within 4 hours, SEV2 within 24 hours, SEV3 within 72 hours.","A post-incident review is completed within 10 business days for SEV1/SEV2.","Critical alerts are triaged within 1 hour."]),
 ("POL-006","Data Classification and Handling Standard","dpo@acmecorp.com","1.8","2026-01-20","2026-01-20",["GRC-DAT-01","GRC-CFG-01"],
  ["Data is classified as Public, Internal or Confidential.","Confidential data must never be stored in publicly accessible locations."]),
 ("POL-007","Encryption Standard","data-protection@acmecorp.com","1.5","2025-11-04","2025-11-04",["GRC-DAT-01"],
  ["Confidential data at rest is encrypted with AES-256 or KMS-managed keys.","Keys are rotated annually."]),
 ("POL-008","Logging and Monitoring Standard","cloud-sec@acmecorp.com","2.2","2026-04-08","2026-04-08",["GRC-LOG-01","GRC-LOG-02","GRC-IAM-07"],
  ["Cloud audit logging is enabled in all accounts and regions and may not be disabled.","Root account usage generates a critical alert."]),
 ("POL-009","Third-Party Risk Management Policy","tprm-lead@acmecorp.com","1.3","2025-09-30","2025-09-30",["GRC-VND-01"],
  ["Critical and High vendors are reassessed every 12 months.","Vendors handling Confidential data must provide a current SOC 2 Type II report."]),
 ("POL-010","Change Management Policy","change-manager@acmecorp.com","3.0","2026-02-24","2026-02-24",["GRC-CFG-02"],
  ["Production changes require an approved change ticket.","Implementers may not approve their own changes."]),
 ("POL-011","Backup and Recovery Standard","infra-lead@acmecorp.com","2.1","2026-06-02","2026-06-02",["GRC-DAT-02"],
  ["Tier 0/1 systems are backed up daily.","Restore tests are performed quarterly for Tier 0/1 systems."]),
 ("POL-012","Security Awareness Policy","awareness-lead@acmecorp.com","1.6","2025-12-09","2025-12-09",["GRC-TRN-01"],
  ["All workforce members complete security awareness training annually.","Phishing simulations are run monthly."]),
]
def etag(*x): return '"' + hashlib.md5("|".join(map(str,x)).encode()).hexdigest()[:8] + ',3"'
def add_year(d): y,m,dd = map(int, d.split("-")); return f"{y+1}-{m:02d}-{dd:02d}"
pol_rows = []
for n, (pid, title, owner, ver, approved, reg_rev, ctrls, stmts) in enumerate(POL, start=1):
    fname = f"{pid}_{title.replace(' ','_')}.pdf"
    pol_rows.append({"id": n, "PolicyId": pid, "Title": title, "PolicyOwner": owner, "Version": ver, "Status": "Published",
        "LastReviewDate": reg_rev, "NextReviewDate": add_year(reg_rev), "MappedControls": ctrls,
        "DocumentUrl": f"{SP_SITE}/Policies/{fname}", "Created": "2023-03-01T10:00:00Z", "Modified": f"{reg_rev}T15:30:00Z",
        "Author": {"Email": "grc-manager@acmecorp.com"}, "Editor": {"Email": owner},
        "_sp_site": SP_SITE, "_sp_list": "Policy Register", "_sp_etag": etag(pid, reg_rev)})

EXC = [  # id, title, policy, control, bu, requester, approver, approval, start, expiry, compensating, risk, list status
 ("EXC-001","Marketing public asset bucket","POL-006","GRC-CFG-01","BU-SAL","marketing-ops@acmecorp.com","ciso@acmecorp.com","Approved","2026-06-01","2026-11-30","Bucket contains only Public-classified assets; weekly content review","RSK-007","Active"),
 ("EXC-002","OT plant hosts without EDR agent","POL-008","GRC-AST-02","BU-OPS","ops-it@acmecorp.com","ciso@acmecorp.com","Approved","2026-07-01","2026-12-31","Network segmentation; host firewall; monthly manual review","RSK-011","Active"),
 ("EXC-003","Legacy ERP integration MFA exemption","POL-002","GRC-IAM-01","BU-FIN","finance-apps@acmecorp.com","ciso@acmecorp.com","Approved","2026-04-01","2026-09-30","IP allow-listing to finance VLAN","RSK-002","Active"),
 ("EXC-004","Mainframe batch account password rotation","POL-002","GRC-IAM-04","BU-FIN","finance-apps@acmecorp.com","iam-lead@acmecorp.com","Approved","2026-03-15","2026-09-15","Vaulted credential; session recording","RSK-002","Active"),
 ("EXC-005","Dev account trail in single region","POL-008","GRC-LOG-01","BU-ENG","platform-eng@acmecorp.com","cloud-sec@acmecorp.com","Approved","2026-05-01","2026-10-31","Org trail covers primary region","RSK-005","Active"),
 ("EXC-006","Windows 2012 file server (EOL) patching","POL-004","GRC-VUL-02","BU-OPS","ops-it@acmecorp.com","ciso@acmecorp.com","Approved","2026-01-10","2026-07-10","Isolated VLAN; extended support agreement","RSK-001","Closed"),
 ("EXC-007","Sales CRM integration service account without MFA","POL-002","GRC-IAM-01","BU-SAL","business-systems@acmecorp.com","iam-lead@acmecorp.com","Approved","2026-08-01","2027-01-31","OAuth client credentials; IP restriction","RSK-002","Active"),
 ("EXC-008","Vendor DataBridge ETL overdue assessment","POL-009","GRC-VND-01","BU-ENG","data-platform@acmecorp.com","tprm-lead@acmecorp.com","Pending","2026-10-01","2027-01-31","Contractual audit clause invoked","RSK-009","Draft"),
 ("EXC-009","Unencrypted analytics scratch buckets","POL-007","GRC-DAT-01","BU-ENG","data-platform@acmecorp.com","data-protection@acmecorp.com","Approved","2026-02-01","2026-08-01","Data limited to synthetic test data","RSK-004","Active"),
 ("EXC-010","Quarterly access review deferral - Engineering","POL-002","GRC-IAM-05","BU-ENG","eng-director@acmecorp.com","iam-lead@acmecorp.com","Rejected","2026-07-15","2026-10-15","None","RSK-003","Closed"),
]
for i in range(11, 26):
    ctrl = ["GRC-VUL-02","GRC-AST-02","GRC-DAT-02","GRC-CFG-02","GRC-IAM-04","GRC-TRN-01","GRC-LOG-02"][i % 7]
    bu = ["BU-ENG","BU-OPS","BU-CORP","BU-FIN","BU-SAL"][i % 5]
    start = (dt.date(2026,3,1) + dt.timedelta(days=(i*17) % 200)).isoformat()
    exp = (dt.date.fromisoformat(start) + dt.timedelta(days=[90,120,150,180][i % 4])).isoformat()
    EXC.append((f"EXC-{i:03d}", f"Temporary deviation for {ctrl} ({bu})", "POL-004" if "VUL" in ctrl else "POL-001", ctrl, bu,
                f"{bu[3:].lower()}-lead@acmecorp.com", "grc-manager@acmecorp.com", "Approved", start, exp, "Compensating monitoring in place",
                ["RSK-001","RSK-006","RSK-010","RSK-012"][i % 4], "Closed" if exp < "2026-09-01" and i % 3 else "Active"))
exc_rows = [{"id": n, "ExceptionId": e[0], "Title": e[1], "PolicyId": e[2], "ControlId": e[3], "BusinessUnit": e[4], "Requester": {"Email": e[5]},
             "Approver": {"Email": e[6]}, "ApprovalStatus": e[7], "StartDate": e[8], "ExpiryDate": e[9], "CompensatingControls": e[10],
             "LinkedRiskId": e[11], "ExceptionStatus": e[12], "Created": f"{e[8]}T09:00:00Z", "Modified": f"{e[8]}T09:00:00Z",
             "Author": {"Email": e[5]}, "_sp_site": SP_SITE, "_sp_list": "Policy Exceptions", "_sp_etag": etag(e[0])} for n, e in enumerate(EXC, start=1)]

RSK = [  # id, title, category, bu, owner, likelihood, impact, ALE, treatment, status, controls, last reviewed
 ("RSK-001","Exploitation of unpatched internet-facing servers","RT-VUL","BU-ENG","vuln-lead@acmecorp.com",4,5,4200000,"Accelerate patching; virtual patching via WAF","In Progress",["GRC-VUL-01","GRC-VUL-02","GRC-VUL-03"],"2026-09-02"),
 ("RSK-002","Account takeover of workforce identities","RT-IAM","BU-CORP","iam-lead@acmecorp.com",3,5,3100000,"Phishing-resistant MFA rollout","In Progress",["GRC-IAM-01","GRC-IAM-02"],"2026-08-20"),
 ("RSK-003","Privilege misuse by over-entitled administrators","RT-IAM","BU-ENG","iam-lead@acmecorp.com",3,4,1800000,"Quarterly recertification; JIT access","Planned",["GRC-IAM-05","GRC-IAM-07"],"2026-06-30"),
 ("RSK-004","Sensitive data exposure through cloud storage misconfiguration","RT-CLD","BU-ENG","cloud-sec@acmecorp.com",3,5,5200000,"CSPM guardrails; SCP blocking public ACLs","Planned",["GRC-CFG-01","GRC-DAT-01"],"2026-07-14"),
 ("RSK-005","Loss of audit trail impairs detection and forensics","RT-CMP","BU-CORP","cloud-sec@acmecorp.com",2,4,900000,"Org-level trail with SCP preventing StopLogging","Implemented",["GRC-LOG-01"],"2026-08-05"),
 ("RSK-006","Ransomware impacting operations","RT-MAL","BU-OPS","soc-manager@acmecorp.com",4,5,7500000,"EDR coverage to 100%; immutable backups","In Progress",["GRC-AST-02","GRC-DAT-02","GRC-IR-01"],"2026-09-18"),
 ("RSK-007","Public website asset tampering","RT-CLD","BU-SAL",None,2,2,120000,"Accept","Accepted",["GRC-CFG-01"],"2026-03-10"),
 ("RSK-008","Orphaned accounts after workforce changes","RT-IAM","BU-CORP","iam-lead@acmecorp.com",3,4,1400000,"HR-driven automated deprovisioning","Planned",["GRC-IAM-03","GRC-IAM-04"],"2026-05-22"),
 ("RSK-009","Critical supplier security failure (payroll, backup, ETL)","RT-TPR","BU-FIN","tprm-lead@acmecorp.com",3,5,2600000,"Reassess critical vendors; contractual remedies","In Progress",["GRC-VND-01"],"2026-07-28"),
 ("RSK-010","Unauthorized production change causes outage","RT-RES","BU-ENG","change-manager@acmecorp.com",3,3,800000,"Automated change detection vs tickets","Planned",["GRC-CFG-02"],"2026-09-25"),
 ("RSK-011","OT/legacy hosts without endpoint protection","RT-MAL","BU-OPS",None,4,4,2100000,"Segmentation; agent rollout to supported OS","In Progress",["GRC-AST-02","GRC-AST-01"],"2026-06-12"),
 ("RSK-012","Phishing leading to credential compromise","RT-IAM","BU-SAL","awareness-lead@acmecorp.com",4,3,950000,"Targeted training for Sales","In Progress",["GRC-TRN-01","GRC-IAM-01"],"2026-09-30"),
 ("RSK-013","Fraudulent vendor payment via SoD conflict","RT-INS","BU-FIN","controller@acmecorp.com",2,5,1600000,"SoD enforcement in ERP","Planned",["GRC-IAM-06"],"2026-04-18"),
 ("RSK-014","Delayed incident containment","RT-RES","BU-CORP","soc-manager@acmecorp.com",3,4,1300000,"SOC staffing; automated containment playbooks","Planned",["GRC-IR-01","GRC-LOG-02"],"2026-08-29"),
 ("RSK-015","Root account compromise in cloud","RT-IAM","BU-FIN","cloud-sec@acmecorp.com",2,5,4800000,"Hardware MFA on root; SCP deny root","In Progress",["GRC-IAM-07"],"2026-07-02"),
 ("RSK-016","Backup failure prevents recovery of Tier 1 systems","RT-RES","BU-CORP","infra-lead@acmecorp.com",2,5,2300000,"Backup monitoring and quarterly restore tests","Implemented",["GRC-DAT-02"],"2026-08-11"),
 ("RSK-017","Outdated policies no longer reflect control environment","RT-CMP","BU-CORP","grc-manager@acmecorp.com",3,2,250000,"Policy review calendar","In Progress",["GRC-POL-01","GRC-POL-02"],"2026-09-05"),
 ("RSK-018","Regulatory fine for PII breach (GDPR/CCPA)","RT-CMP","BU-CORP","dpo@acmecorp.com",2,5,6000000,"Data minimization; DLP","Planned",["GRC-DAT-01","GRC-CFG-01"],"2026-06-01"),
 ("RSK-019","Insider data exfiltration by departing employees","RT-INS","BU-SAL",None,3,3,700000,"UEBA monitoring of leavers","Planned",["GRC-IAM-03"],"2026-02-14"),
 ("RSK-020","Shadow IT SaaS adoption","RT-DAT","BU-ENG","cloud-sec@acmecorp.com",3,2,300000,"CASB discovery","Planned",["GRC-AST-01"],"2026-08-15"),
]
rsk_rows = [{"id": n, "RiskId": r[0], "Title": r[1], "RiskCategoryId": r[2], "BusinessUnit": r[3], "RiskOwner": ({"Email": r[4]} if r[4] else None),
             "Likelihood": r[5], "Impact": r[6], "InherentScore": r[5]*r[6], "AnnualLossExpectancyUSD": r[7], "TreatmentPlan": r[8], "TreatmentStatus": r[9],
             "LinkedControls": r[10], "LastReviewed": r[11], "Created": "2025-01-15T10:00:00Z", "Modified": f"{r[11]}T12:00:00Z",
             "Author": {"Email": "grc-manager@acmecorp.com"}, "_sp_site": SP_SITE, "_sp_list": "Risk Register", "_sp_etag": etag(r[0], r[11])} for n, r in enumerate(RSK, start=1)]

# small config-style lists written as JSON lines
for name, rows in [("policy_register", pol_rows), ("policy_exceptions", exc_rows), ("risk_register", rsk_rows)]:
    os.makedirs(f"{SP}/lists/{name}", exist_ok=True)
    with open(f"{SP}/lists/{name}/{name}_2026-10-27.json", "w") as f:
        for r in rows: f.write(json.dumps(r) + "\n")

# policy documents (PDF) + driveItem metadata
os.makedirs(f"{SP}/documents/Policies", exist_ok=True)
meta = []
for pid, title, owner, ver, approved, reg_rev, ctrls, stmts in POL:
    pdf = FPDF(); pdf.add_page(); pdf.set_auto_page_break(True, 15)
    pdf.set_font("Helvetica", "B", 18); pdf.cell(0, 12, "ACME Corp", new_x="LMARGIN", new_y="NEXT")
    pdf.set_font("Helvetica", "B", 15); pdf.multi_cell(0, 9, title, new_x="LMARGIN", new_y="NEXT"); pdf.ln(2)
    for k, v in [("Document ID", pid), ("Version", ver), ("Classification", "Internal"), ("Policy owner", owner),
                 ("Approved by", "Chief Information Security Officer"), ("Approval date", approved), ("Next review date", add_year(approved))]:
        pdf.set_font("Helvetica", "B", 10); pdf.cell(45, 7, k, border=1); pdf.set_font("Helvetica", "", 10); pdf.cell(0, 7, v, border=1, new_x="LMARGIN", new_y="NEXT")
    pdf.ln(4)
    topic = title.lower().replace(' policy','').replace(' standard','').replace(' plan','')
    for head, body in [("1. Purpose", f"This document defines ACME Corp requirements for {topic} to protect the confidentiality, integrity and availability of company information."),
                       ("2. Scope", "Applies to all business units, workforce members, contractors, systems and third parties processing ACME Corp information.")]:
        pdf.set_font("Helvetica", "B", 12); pdf.cell(0, 8, head, new_x="LMARGIN", new_y="NEXT"); pdf.set_font("Helvetica", "", 10); pdf.multi_cell(0, 6, body, new_x="LMARGIN", new_y="NEXT"); pdf.ln(2)
    pdf.set_font("Helvetica", "B", 12); pdf.cell(0, 8, "3. Policy statements", new_x="LMARGIN", new_y="NEXT"); pdf.set_font("Helvetica", "", 10)
    for j, s in enumerate(stmts, 1): pdf.multi_cell(0, 6, f"3.{j}  {s}", new_x="LMARGIN", new_y="NEXT")
    pdf.ln(2); pdf.set_font("Helvetica", "B", 12); pdf.cell(0, 8, "4. Related controls", new_x="LMARGIN", new_y="NEXT"); pdf.set_font("Helvetica", "", 10)
    pdf.multi_cell(0, 6, "Unified control set: " + ", ".join(ctrls), new_x="LMARGIN", new_y="NEXT")
    pdf.ln(2); pdf.set_font("Helvetica", "B", 12); pdf.cell(0, 8, "5. Compliance and exceptions", new_x="LMARGIN", new_y="NEXT"); pdf.set_font("Helvetica", "", 10)
    pdf.multi_cell(0, 6, "Non-compliance must be reported to the GRC team. Exceptions must be requested via the Policy Exceptions list, approved, time-bound and may not exceed 180 days.", new_x="LMARGIN", new_y="NEXT")
    fname = f"{pid}_{title.replace(' ','_')}.pdf"; path = f"{SP}/documents/Policies/{fname}"
    pdf.output(path)
    meta.append({"id": "01" + hashlib.md5(pid.encode()).hexdigest()[:24].upper(), "name": fname, "webUrl": f"{SP_SITE}/Policies/{fname}",
                 "size": os.path.getsize(path), "createdDateTime": "2023-03-01T10:00:00Z", "lastModifiedDateTime": f"{approved}T15:00:00Z",
                 "createdBy": {"user": {"email": "grc-manager@acmecorp.com"}}, "lastModifiedBy": {"user": {"email": owner}},
                 "file": {"mimeType": "application/pdf"}, "parentReference": {"path": "/drives/Policies/root:"}, "_sp_site": SP_SITE,
                 "_sp_library": "Policies", "_volume_path": path})
os.makedirs(f"{SP}/documents_metadata", exist_ok=True)
with open(f"{SP}/documents_metadata/policies_driveitems_2026-10-27.json", "w") as f:
    for m in meta: f.write(json.dumps(m) + "\n")

In [0]:
csv = lambda p: spark.read.option("header", True).option("escape", '"').csv(f"{LAND}/{p}")
js = lambda p: spark.read.json(f"{LAND}/{p}")
hr, cmdb = csv("hr/workers_snapshot_2026-10-27"), csv("servicenow/cmdb_ci/snapshot_2026-10-27")
users, mem, sl = js("okta/users"), spark.read.parquet(f"{LAND}/okta/group_memberships"), js("okta/system_log")
ct = js("aws/cloudtrail").select(F.explode("Records").alias("r")).select("r.*")
cfg, vul, dev, al = js("aws/config_snapshot/2026-10-27"), csv("tenable/vuln_export"), js("crowdstrike/devices/2026-10-27"), js("crowdstrike/alerts")
chg, inc, vit = csv("servicenow/change_request"), csv("servicenow/sn_si_incident"), csv("servicenow/sn_vul_vulnerable_item")
enr, ph, vnd = csv("lms/enrollments"), csv("lms/phishing_results"), csv("tprm/vendor_assessments")
pol, exc, rsk = js("sharepoint/lists/policy_register"), js("sharepoint/lists/policy_exceptions"), js("sharepoint/lists/risk_register")
print({n: d.count() for n, d in [("hr_workers",hr),("cmdb_ci",cmdb),("okta_users",users),("okta_memberships",mem),("okta_signins",sl),("cloudtrail",ct),
       ("aws_config",cfg),("tenable_rows",vul),("edr_devices",dev),("edr_alerts",al),("sn_changes",chg),("sn_incidents",inc),("sn_vits",vit),
       ("lms_enrollments",enr),("lms_phishing",ph),("vendors",vnd),("sp_policies",pol),("sp_exceptions",exc),("sp_risks",rsk)]})
term = hr.filter("employment_status='Terminated'").select(F.lower("email").alias("login"), "termination_date")
act = users.filter("status='ACTIVE'").select(F.col("profile.login").alias("login"), "id", F.col("profile.department").alias("department"))
checks = {
 "FK: memberships->users (expect 0)": mem.join(users, mem.user_id == users.id, "left_anti").count(),
 "FK: cmdb owner->hr (expect 0)": cmdb.filter("owned_by IS NOT NULL").join(hr, cmdb.owned_by == hr.email, "left_anti").count(),
 "FK: vuln host->cmdb (expect 0)": vul.select("hostname").distinct().join(cmdb, F.col("hostname") == cmdb.name, "left_anti").count(),
 "FK: edr host->cmdb (expect 0)": dev.join(cmdb, dev.hostname == cmdb.name, "left_anti").count(),
 "Orphaned (terminated HR, active IdP)": term.join(act, "login").count(),
 "Orphans signing in after termination": sl.join(term, F.col("actor.alternateId") == term.login).filter(F.to_date("published") > F.col("termination_date")).select("actor.alternateId").distinct().count(),
 "Active workforce users w/o MFA factor": users.filter("status='ACTIVE' AND profile.userType='workforce' AND size(factors)=0").count(),
 "FIN no-MFA sign-ins after Oct-5": sl.filter("event_date >= '2026-10-05' AND NOT authenticationContext.mfaVerified").join(act, F.col("actor.id") == act.id).filter("department='BU-FIN'").count(),
 "Root logins without MFA": ct.filter("userIdentity.type='Root' AND eventName='ConsoleLogin' AND additionalEventData.MFAUsed='No'").count(),
 "StopLogging events": ct.filter("eventName='StopLogging'").count(),
 "Public buckets in config": cfg.filter("resourceType='AWS::S3::Bucket' AND configuration.aclGrantsAllUsers").count(),
 "Trails not logging": cfg.filter("resourceType='AWS::CloudTrail::Trail' AND NOT configuration.isLogging").count(),
 "CVE-2026-31337 active on last scan": vul.filter("scan_date='2026-10-26' AND cve='CVE-2026-31337' AND state='Active'").count(),
 "Installed servers/endpoints w/o EDR": cmdb.filter("install_status='Installed' AND sys_class_name IN ('cmdb_ci_linux_server','cmdb_ci_win_server','cmdb_ci_computer')").join(dev, cmdb.name == dev.hostname, "left_anti").count(),
 "Story EDR alerts (INC0012045)": al.filter("linked_incident='INC0012045'").count(),
 "Policies overdue per register": pol.filter("NextReviewDate < '2026-10-28'").count(),
 "Active exceptions past expiry": exc.filter("ExceptionStatus='Active' AND ExpiryDate < '2026-10-28'").count(),
 "Risks without owner": rsk.filter("RiskOwner IS NULL").count(),
 "Critical vendors assessment >365d": vnd.filter("criticality='Critical' AND last_assessment_date < '2025-10-27'").count(),
}
for k, v in checks.items(): print(f"{k}: {v}")